# M5 corrected source and environment preparation — CPU

1. Create a **private Kaggle dataset** named `m5-kaggle-source-v2`. Upload only
   **`m5-kaggle-source.payload`**, preserving that filename and extension.
2. Import this notebook as a **private notebook**. Set **Accelerator: None** and
   **Internet: On**. Use Add Input to attach the new dataset. The older dataset
   is unnecessary; keeping it attached is supported for input diagnosis.
3. Select **Run All**. Defaults need no cell edits. Installation and checks may
   take several minutes; the CPU workflow has a 45-minute limit.
4. Download `preflight-results.zip` from the final cell and the executed notebook.
   Stop the session. Save both files under your local `notebooks/run/3/` and return them.

This verifies every canonical source member, stages 4.07 GB of pinned CUDA wheels
on disposable scratch, creates a clean candidate venv, checks installed package
provenance, and runs 81 focused plus 16 native checks. The installed-package probe
runs separately outside the source directory. The active Kaggle kernel is unchanged.

No language weights are downloaded and no language-model generations occur.
This stage uses CPU with GPU-compatible packages. A passing result still requires
review before GPU library probes, model staging or bounded smoke. Accelerator None
is required so a cold installation cannot consume the short GPU allocation window.
Both passing checks and handled failures produce diagnostic evidence.


In [1]:
import hashlib,json,os
from pathlib import Path

PREFLIGHT_PROGRAM='"""Bounded CPU source/environment preparation; never load weights or run models."""\n\nfrom __future__ import annotations\n\nfrom contextlib import closing\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport platform\nimport re\nimport shutil\nimport signal\nimport sqlite3\nimport stat\nimport subprocess\nimport sys\nimport time\nimport uuid\nimport zipfile\n\n\nPINNED = {"torch": "2.11.0+cu128", "pydantic": "2.13.5", "langgraph": "1.2.12",\n          "langgraph-checkpoint": "4.2.0", "langgraph-checkpoint-sqlite": "3.1.1",\n          "langchain-core": "1.6.6", "transformers": "5.12.0", "accelerate": "1.14.0",\n          "bitsandbytes": "0.49.2", "tokenizers": "0.22.2", "huggingface-hub": "1.19.0"}\nINDEX_ARGS = ["--index-url", "https://pypi.org/simple", "--extra-index-url",\n              "https://download.pytorch.org/whl/cu128"]\nOUTPUT_RESERVE = 256 * 2**20\nBUILD_RESERVE = 2**30\nCPU_SECONDS = 2700\nEXPORT_SECONDS = 90\nINVENTORY_LIMIT = 2000\nFOCUSED = ["tests.test_inference_preparation", "tests.inference_checks", "tests.durable_checks",\n           "tests.attempt_checks", "tests.worker_checks"]\nSTAGES = ["input_inventory", "source", "cpu_host", "command_cleanup", "environment", "installed_packages",\n          "focused_checks", "native_9b", "native_4b", "native_capacity", "filesystem_backup"]\nMODEL_SELECTED_BYTES = {"9B": 19319226111, "4B": 9332739901}\n\n\ndef utc() -> str:\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef sha256_file(path: Path) -> str:\n    with path.open("rb") as stream:\n        return hashlib.file_digest(stream, "sha256").hexdigest()\n\n\ndef save_json(path: Path, value: object) -> None:\n    temporary = path.with_suffix(path.suffix + ".partial")\n    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\\n", encoding="utf-8")\n    temporary.replace(path)\n\n\ndef inventory(root: Path, limit: int = INVENTORY_LIMIT) -> dict:\n    entries, errors, pending = [], [], [root]\n    truncated = False\n    if root.is_symlink() or not root.is_dir():\n        raise ValueError(f"input_root_missing_or_symlink: {root}")\n    while pending:\n        directory = pending.pop()\n        try:\n            with os.scandir(directory) as children:\n                for child in children:\n                    if len(entries) >= limit:\n                        truncated = True\n                        break\n                    info = child.stat(follow_symlinks=False)\n                    kind = ("symlink" if child.is_symlink() else "directory" if stat.S_ISDIR(info.st_mode)\n                            else "file" if stat.S_ISREG(info.st_mode) else "other")\n                    entries.append({"path": Path(child.path).relative_to(root).as_posix(),\n                                    "kind": kind, "bytes": info.st_size})\n                    if kind == "directory":\n                        pending.append(Path(child.path))\n        except OSError as exc:\n            errors.append({"path": str(directory), "reason": str(exc)})\n        if truncated:\n            break\n    return {"root": str(root), "entries": sorted(entries, key=lambda x: x["path"]),\n            "limit": limit, "truncated": truncated, "errors": errors}\n\n\ndef transformed_sources(root: Path, listing: dict) -> list[dict]:\n    entries = {x["path"]: x for x in listing["entries"]}\n    result = []\n    for name, item in entries.items():\n        if item["kind"] != "file" or not name.endswith("SOURCE_COMMIT.txt") or item["bytes"] > 128:\n            continue\n        source = PurePosixPath(name).parent\n        paths = {}\n        for fixture in ("capacity_report.json.gz", "capacity_report.json",\n                        "scripted_lifecycle_run.zip", "scripted_lifecycle_run"):\n            relative = (source / "tests/fixtures" / fixture).as_posix()\n            paths[fixture] = entries.get(relative, {"kind": "missing"})\n        result.append({"root": str(root / source), "paths": paths})\n    return result\n\n\ndef safe_members(archive: zipfile.ZipFile, expected: dict) -> list[zipfile.ZipInfo]:\n    infos = archive.infolist()\n    names = []\n    for entry in infos:\n        name = entry.orig_filename\n        if (not name or name != entry.filename or "\\\\" in name or ":" in name\n                or name.startswith("/") or any(ord(c) < 32 for c in name)\n                or any(part in {"", ".", ".."} for part in name.split("/"))\n                or entry.is_dir() or stat.S_ISLNK(entry.external_attr >> 16)):\n            raise ValueError(f"unsafe_archive_member: {name!r}")\n        if name in names or any(str(p) in names for p in PurePosixPath(name).parents):\n            raise ValueError(f"duplicate_or_conflicting_member: {name}")\n        if any(other.startswith(name + "/") for other in names):\n            raise ValueError(f"conflicting_member: {name}")\n        names.append(name)\n    if set(names) != set(expected):\n        raise ValueError(f"archive_member_set: missing={sorted(set(expected)-set(names))}; "\n                         f"unexpected={sorted(set(names)-set(expected))}")\n    for entry in infos:\n        if entry.file_size != expected[entry.filename]["bytes"]:\n            raise ValueError(f"archive_member_size: {entry.filename}")\n    return infos\n\n\ndef admit_source(payload: Path, destination: Path, expected: dict) -> dict:\n    if expected["expanded_bytes"] != sum(x["bytes"] for x in expected["members"].values()):\n        raise ValueError("source_manifest_expanded_bytes")\n    if payload.is_symlink() or not payload.is_file():\n        raise ValueError("payload_not_regular")\n    if payload.stat().st_size != expected["archive_bytes"]:\n        raise ValueError("payload_size_mismatch")\n    if sha256_file(payload) != expected["archive_sha256"]:\n        raise ValueError("payload_hash_mismatch")\n    if destination.exists():\n        raise FileExistsError("source_destination_exists")\n    with zipfile.ZipFile(payload) as archive:\n        infos = safe_members(archive, expected["members"])\n        # Validate every member before executing/importing/copying any source.\n        for entry in infos:\n            with archive.open(entry) as stream:\n                digest = hashlib.file_digest(stream, "sha256").hexdigest()\n            if digest != expected["members"][entry.filename]["sha256"]:\n                raise ValueError(f"archive_member_hash: {entry.filename}")\n        marker = archive.read("SOURCE_COMMIT.txt").decode("ascii").strip()\n        if marker != expected["source_baseline_commit"]:\n            raise ValueError("source_baseline_marker")\n        destination.mkdir(parents=True, exist_ok=False)\n        for entry in infos:\n            target = destination / entry.filename\n            if not target.resolve().is_relative_to(destination.resolve()):\n                raise ValueError(f"source_path_escape: {entry.filename}")\n            target.parent.mkdir(parents=True, exist_ok=True)\n            with archive.open(entry) as incoming, target.open("xb") as output:\n                shutil.copyfileobj(incoming, output, 1024 * 1024)\n            if sha256_file(target) != expected["members"][entry.filename]["sha256"]:\n                raise ValueError(f"extracted_member_hash: {entry.filename}")\n    return {"payload": str(payload), "archive_sha256": expected["archive_sha256"],\n            "verified_members": len(infos), "expanded_bytes": expected["expanded_bytes"],\n            "baseline_commit": marker, "build_commit": expected.get("build_commit"),\n            "dataset_version": None, "dataset_reference": str(payload.parent),\n            "closure": "Every canonical archive member, including unchanged compressed fixtures."}\n\n\ndef filesystem_facts(paths: dict[str, Path]) -> dict:\n    locations, pools = {}, {}\n    for role, path in paths.items():\n        info, disk = path.stat(), shutil.disk_usage(path)\n        key = str(info.st_dev)\n        locations[role] = {"path": str(path), "pool": key, "free_bytes": disk.free,\n                           "total_bytes": disk.total}\n        pools[key] = {"free_bytes": min(disk.free, pools.get(key, {}).get("free_bytes", disk.free)),\n                      "roles": pools.get(key, {}).get("roles", []) + [role]}\n    return {"locations": locations, "pools": pools, "quota_verified": False,\n            "interpretation": "Filesystem free space is observed; shared pools are counted once."}\n\n\ndef require_space(scratch: Path, output: Path, additional_scratch: int) -> dict:\n    facts = filesystem_facts({"scratch": scratch, "output": output})\n    same = scratch.stat().st_dev == output.stat().st_dev\n    if shutil.disk_usage(output).free < OUTPUT_RESERVE:\n        raise ValueError("output_evidence_reserve")\n    required = additional_scratch + (OUTPUT_RESERVE if same else 0)\n    scratch_pool=facts[\'locations\'][\'scratch\'][\'pool\']\n    if facts[\'pools\'][scratch_pool][\'free_bytes\'] < required:\n        raise ValueError(f"scratch_capacity: additional_required={required}")\n    facts.update(additional_scratch_bytes=additional_scratch, output_reserve_bytes=OUTPUT_RESERVE)\n    return facts\n\n\ndef _group_exists(pid: int, proc_root: Path = Path(\'/proc\')) -> bool:\n    try:\n        os.killpg(pid, 0)\n    except ProcessLookupError:\n        return False\n    if not proc_root.is_dir():\n        return True  # Presence is known; live membership cannot be inspected.\n    for path in proc_root.iterdir():\n        if not path.name.isdigit():\n            continue\n        try:\n            fields=(path/\'stat\').read_text().rsplit(\')\',1)[1].split()\n            if int(fields[2])==pid and fields[0] not in {\'Z\',\'X\'}:\n                return True\n        except FileNotFoundError:\n            continue  # Process exited during the read.\n        except (OSError,ValueError,IndexError):\n            return True  # Never claim cleanup from an incomplete inspection.\n    return False  # Zombies have exited and hold no executable/CUDA state.\n\n\ndef _stop_owned(process: subprocess.Popen) -> dict:\n    if os.name == "nt":\n        result = subprocess.run(["taskkill", "/PID", str(process.pid), "/T", "/F"],\n                                capture_output=True, timeout=5)\n        try:\n            process.wait(timeout=3)\n        except subprocess.TimeoutExpired:\n            process.kill()\n            process.wait(timeout=3)\n        return {"leader_exited": process.poll() is not None,\n                "tree_confirmation": "taskkill" if result.returncode == 0 else "unconfirmed"}\n    for sig in (signal.SIGTERM, signal.SIGKILL):\n        if not _group_exists(process.pid):\n            break\n        try:\n            os.killpg(process.pid, sig)\n        except ProcessLookupError:\n            break\n        end = time.monotonic() + 2\n        while time.monotonic() < end:\n            process.poll()\n            if not _group_exists(process.pid):\n                break\n            time.sleep(0.05)\n    process.wait(timeout=2)\n    return {"leader_exited": True,\n            "tree_confirmation": "exited" if not _group_exists(process.pid) else "unconfirmed"}\n\n\ndef command(output: Path, name: str, args: list, *, deadline: float, timeout: float,\n            cwd: Path, env: dict) -> dict:\n    remaining = deadline - time.monotonic() - EXPORT_SECONDS\n    if remaining < timeout + 7:\n        raise TimeoutError(f"deadline_before_command: {name}, remaining={remaining:.1f}, bound={timeout}")\n    logs = output / "logs"\n    logs.mkdir(exist_ok=True)\n    stdout, stderr = logs / (name + ".stdout.txt"), logs / (name + ".stderr.txt")\n    record = {"name": name, "args": list(map(str, args)), "cwd": str(cwd), "started_utc": utc(),\n              "timeout_seconds": timeout, "stdout": stdout.relative_to(output).as_posix(),\n              "stderr": stderr.relative_to(output).as_posix(), "status":"failed",\n              "leader_exited":False,"tree_confirmation":"not_started"}\n    started, process = time.monotonic(), None\n    try:\n        with stdout.open("xb") as out, stderr.open("xb") as err:\n            options = {"creationflags": subprocess.CREATE_NEW_PROCESS_GROUP} if os.name == "nt" else {\n                "start_new_session": True}\n            process = subprocess.Popen(list(map(str, args)), cwd=cwd, env=env, stdout=out, stderr=err,\n                                       stdin=subprocess.DEVNULL, **options)\n            record["pid"] = process.pid\n            record[\'tree_confirmation\']=\'unconfirmed\'\n            try:\n                record["exit_code"] = process.wait(timeout=timeout)\n                record["status"] = "passed" if process.returncode == 0 else "failed"\n                record["leader_exited"] = True\n                if os.name != "nt" and _group_exists(process.pid):\n                    record.update(_stop_owned(process), status="failed", reason="descendants_survived")\n                elif os.name != "nt":\n                    record[\'tree_confirmation\']=\'exited\'\n                else:\n                    record[\'tree_confirmation\']=\'not_checked_on_windows\'\n            except BaseException as exc:\n                record.update(status="failed", reason=type(exc).__name__, **_stop_owned(process))\n                raise\n    finally:\n        record.update(ended_utc=utc(), elapsed_seconds=time.monotonic()-started)\n        save_json(logs / (name + ".json"), record)\n    if record["status"] != "passed":\n        raise RuntimeError(f"command_failed: {name}, exit={record.get(\'exit_code\')}, "\n                           f"reason={record.get(\'reason\')}; see logs/{name}.stderr.txt")\n    return record\n\n\ndef json_stdout(output: Path, record: dict) -> object:\n    path = output / record["stdout"]\n    if path.stat().st_size > 2 * 2**20:\n        raise ValueError(f"machine_stdout_too_large: {record[\'name\']}")\n    return json.loads(path.read_text(encoding="utf-8"))\n\n\ndef process_tree_diagnostic(output: Path, scratch: Path, deadline: float, env: dict) -> dict:\n    ticker=scratch/\'process-tree-ticks\'\n    grandchild="import pathlib,time;p=pathlib.Path("+repr(str(ticker))+");[(p.write_text(str(n)),time.sleep(.05)) for n in range(600)]"\n    child="import subprocess,sys,time;subprocess.Popen([sys.executable,\'-c\',"+repr(grandchild)+"]);time.sleep(30)"\n    parent="import subprocess,sys,time;subprocess.Popen([sys.executable,\'-c\',"+repr(child)+"]);print(\'owned-tree-started\',flush=True);time.sleep(30)"\n    try:\n        command(output,\'owned-process-timeout\',[sys.executable,\'-c\',parent],deadline=deadline,\n                timeout=2,cwd=scratch,env=env)\n    except subprocess.TimeoutExpired:\n        record=json.loads((output/\'logs/owned-process-timeout.json\').read_text())\n    else:\n        raise ValueError(\'owned_tree_expected_timeout_missing\')\n    confirmation=\'taskkill\' if os.name==\'nt\' else \'exited\'\n    if not record.get(\'leader_exited\') or record.get(\'tree_confirmation\')!=confirmation or not ticker.is_file():\n        raise ValueError(\'owned_tree_cleanup_unconfirmed\')\n    before=ticker.read_text()\n    time.sleep(.2)\n    if ticker.read_text()!=before:\n        raise ValueError(\'owned_grandchild_still_running\')\n    return {\'expected_timeout\':True,\'leader_exited\':True,\'tree_confirmation\':confirmation,\n            \'grandchild_stopped\':True,\'scope\':\'Synthetic CPU process diagnostic; no model or GPU initialization.\'}\n\n\ndef test_result(output: Path, record: dict, expected_count: int) -> dict:\n    text = (output / record["stderr"]).read_text(encoding="utf-8")\n    matches = re.findall(r"^Ran (\\d+) tests? in ([\\d.]+)s$", text, flags=re.MULTILINE)\n    ok = re.search(r"^OK(?: \\(([^\\n]+)\\))?\\s*$", text, flags=re.MULTILINE)\n    skipped = re.search(r"skipped=(\\d+)", ok.group(1) or "") if ok else None\n    if len(matches) != 1 or not ok or int(matches[0][0]) != expected_count or skipped:\n        raise ValueError(f"test_count_or_skip: {record[\'name\']}, expected={expected_count}")\n    return {"tests": expected_count, "skipped": 0, "reported_seconds": float(matches[0][1]),\n            "scope": "Source-mode candidate-host checks; no physical model generation."}\n\n\ndef sqlite_diagnostic(output: Path) -> dict:\n    database, backup = output / "filesystem.sqlite", output / "filesystem-backup.sqlite"\n    with closing(sqlite3.connect(database, timeout=1)) as db:\n        journal = db.execute("PRAGMA journal_mode=WAL").fetchone()[0]\n        db.execute("PRAGMA synchronous=FULL")\n        synchronous = db.execute("PRAGMA synchronous").fetchone()[0]\n        db.execute("CREATE TABLE probe(value INTEGER NOT NULL)")\n        db.execute("INSERT INTO probe VALUES (7)")\n        db.commit()\n        with closing(sqlite3.connect(backup, timeout=1)) as copy:\n            db.backup(copy)\n    with closing(sqlite3.connect(backup.as_uri()+"?mode=ro&immutable=1", uri=True, timeout=1)) as db:\n        integrity = db.execute("PRAGMA integrity_check").fetchone()[0]\n        rows = db.execute("SELECT value FROM probe").fetchall()\n        try:\n            db.execute("INSERT INTO probe VALUES (8)")\n        except sqlite3.OperationalError:\n            read_only = True\n        else:\n            read_only = False\n    if journal != "wal" or synchronous != 2 or integrity != "ok" or rows != [(7,)] or not read_only:\n        raise ValueError("filesystem_sqlite_diagnostic")\n    return {"journal_mode": journal, "synchronous": synchronous, "integrity": integrity,\n            "rows": rows, "read_only_enforced": read_only, "backup_sha256": sha256_file(backup),\n            "scope": "Small synthetic diagnostic; actual worker/crash checks are separate."}\n\n\ndef candidate_environment(scratch: Path) -> dict:\n    env = {k: v for k, v in os.environ.items()\n           if not k.startswith(("PYTHON", "PIP_", "HF_", "TRANSFORMERS_", "VIRTUAL_ENV",\n                                "LANGSMITH_", "LANGCHAIN_"))}\n    env.update(PYTHONNOUSERSITE="1", PYTHONDONTWRITEBYTECODE="1", PYTHONUTF8="1",\n               TOKENIZERS_PARALLELISM="false", HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1",\n               HF_HUB_CACHE=str(scratch/"hub"), TMPDIR=str(scratch/"tmp"), TEMP=str(scratch/"tmp"),\n               TMP=str(scratch/"tmp"), PIP_CONFIG_FILE=os.devnull)\n    (scratch/"tmp").mkdir(exist_ok=True)\n    return env\n\n\ndef installed_probe_script(expected: dict, versions: dict) -> str:\n    hashes = {n: item["sha256"] for n, item in expected["members"].items()\n              if n.startswith("recovery/") and n.endswith(".py")}\n    return "\\n".join([\n        "import hashlib,json,sys,site", "from pathlib import Path",\n        "from importlib.metadata import version", "import recovery,recovery.inference,recovery.runner",\n        "from langgraph.checkpoint.sqlite import SqliteSaver",\n        "from transformers import Qwen3_5ForCausalLM", "import torch",\n        f"wanted={versions!r}", "actual={n:version(n) for n in wanted}",\n        "assert actual==wanted, {\'expected\':wanted,\'actual\':actual}",\n        "assert sys.prefix!=sys.base_prefix and site.ENABLE_USER_SITE is False",\n        "assert torch.__version__==\'2.11.0+cu128\' and torch.version.cuda==\'12.8\'",\n        "package=Path(recovery.__file__).resolve().parent", f"expected={hashes!r}",\n        "observed={\'recovery/\'+p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in package.glob(\'*.py\')}",\n        "assert observed==expected, \'source-to-installed module bytes differ\'",\n        "assert package.is_relative_to(Path(sys.prefix).resolve()), \'base/source import contamination\'",\n        "print(json.dumps({\'versions\':actual,\'executable\':sys.executable,\'prefix\':sys.prefix,",\n        "\'sys_path\':sys.path,\'module_root\':str(package),\'verified_module_hashes\':observed,",\n        "\'qwen_text_class\':Qwen3_5ForCausalLM.__name__,\'torch_cuda_build\':torch.version.cuda,",\n        "\'gpu_native_diagnostics\':None,\'model_generations\':0}))", ""])\n\n\ndef export_evidence(output: Path) -> dict:\n    paths = sorted(p for p in output.rglob("*") if p.is_file() and not p.is_symlink()\n                   and (p.suffix in {".json", ".txt", ".lock", ".sqlite"}))\n    partial, final = output/"preflight-results.zip.partial", output/"preflight-results.zip"\n    with zipfile.ZipFile(partial, "x", compression=zipfile.ZIP_DEFLATED) as archive:\n        for path in paths:\n            archive.write(path, path.relative_to(output).as_posix())\n    with zipfile.ZipFile(partial) as archive:\n        if archive.testzip() is not None or archive.namelist() != [p.relative_to(output).as_posix() for p in paths]:\n            raise ValueError("export_crc_or_members")\n    partial.replace(final)\n    return {"zip_path": str(final), "zip_sha256": sha256_file(final), "zip_bytes": final.stat().st_size}\n\n\ndef run_preflight(input_root: Path, output_root: Path, scratch_root: Path, expected: dict,\n                  lock_text: str, *, workflow_seconds: float = CPU_SECONDS) -> dict:\n    started = time.monotonic()\n    deadline = started + workflow_seconds\n    label = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")+"-"+uuid.uuid4().hex[:8]\n    output = output_root/(\'recovery-m5-preflight-\'+label)\n    output.mkdir(parents=True, exist_ok=False)\n    scratch = scratch_root/label\n    source, candidate = scratch/"source", scratch/"venv"\n    report = {"schema_version": 2, "kind": "cpu-canonical-source-preparation", "started_utc": utc(),\n              "output_directory": str(output), "scratch_directory": str(scratch),\n              "cpu_workflow_limit_seconds": workflow_seconds, "export_reserve_seconds": EXPORT_SECONDS,\n              "stages": {n: {"status": "not_run", "reason": None} for n in STAGES}, "root_blocker": None}\n    payload, listing, env = None, None, None\n\n    def stage(name, prerequisites, action):\n        state = report["stages"][name]\n        state["prerequisites"] = prerequisites\n        blocked = [p for p in prerequisites if report["stages"][p]["status"] != "passed"]\n        if blocked:\n            state.update(status="blocked", reason="prerequisites: "+", ".join(blocked))\n        elif deadline-time.monotonic() < EXPORT_SECONDS:\n            state.update(status="blocked", reason="cpu_deadline_before_stage")\n            report["root_blocker"] = report["root_blocker"] or "cpu_deadline_before_stage"\n        else:\n            then = time.monotonic()\n            state.update(started_utc=utc(), status="running")\n            save_json(output/"preflight-report.json", report)\n            try:\n                state.update(result=action(), status="passed")\n            except KeyboardInterrupt:\n                state.update(status="failed", reason="user_interruption")\n                report["root_blocker"] = report["root_blocker"] or "user_interruption"\n                raise\n            except Exception as exc:\n                state.update(status="failed", reason=f"{type(exc).__name__}: {exc}")\n                report["root_blocker"] = report["root_blocker"] or state["reason"]\n            finally:\n                state.update(ended_utc=utc(), elapsed_seconds=time.monotonic()-then)\n        save_json(output/"preflight-report.json", report)\n        print(f"{name}: {state[\'status\']}"+(f" — {state[\'reason\']}" if state["reason"] else ""), flush=True)\n\n    def input_stage():\n        nonlocal payload, listing\n        listing = inventory(input_root)\n        save_json(output/"input-inventory.json", listing)\n        save_json(output/"source-input-diagnosis.json", {"transformed_sources": transformed_sources(input_root, listing)})\n        if listing["truncated"] or listing["errors"]:\n            raise ValueError("incomplete_input_inventory")\n        choices = [input_root/x["path"] for x in listing["entries"]\n                   if x["kind"] == "file" and PurePosixPath(x["path"]).name == expected["payload_filename"]]\n        if len(choices) != 1:\n            raise ValueError("payload_missing" if not choices else "payload_ambiguous")\n        payload = choices[0]\n        return {"selected_payload": str(payload), "entries": len(listing["entries"])}\n\n    def source_stage():\n        scratch.mkdir(parents=True, exist_ok=False)\n        facts = require_space(scratch, output, expected["expanded_bytes"]+BUILD_RESERVE)\n        save_json(output/"storage-before-source.json", facts)\n        return admit_source(payload, source, expected)\n\n    def host_stage():\n        gpu_dir = Path("/proc/driver/nvidia/gpus")\n        gpus = sorted(p.name for p in gpu_dir.iterdir()) if gpu_dir.is_dir() else []\n        host = {"python": platform.python_version(), "os": platform.platform(),\n                "sqlite": sqlite3.sqlite_version, "visible_driver_gpu_entries": gpus,\n                "cuda_initialized_by_kernel": False}\n        save_json(output/"host.json", host)\n        if gpus:\n            raise ValueError("cpu_stage_gpu_visible: stop session and select Accelerator None")\n        if platform.system() != "Linux" or sys.version_info[:2] != (3, 13) or platform.machine() != "x86_64":\n            raise ValueError("candidate_host_requires_linux_x86_64_python313")\n        return host\n\n    def invoke(name, args, timeout, *, cwd=None, source_mode=False):\n        if report.get(\'process_cleanup_unconfirmed\'):\n            raise RuntimeError(\'prior_process_cleanup_unconfirmed\')\n        require_space(scratch, output, BUILD_RESERVE)\n        process_env = dict(env)\n        if source_mode:\n            process_env["PYTHONPATH"] = str(source)\n        try:\n            return command(output, name, args, deadline=deadline, timeout=timeout,\n                           cwd=cwd or scratch, env=process_env)\n        except BaseException:\n            record_path = output/\'logs\'/(name+\'.json\')\n            if record_path.exists() and json.loads(record_path.read_text()).get(\'tree_confirmation\')==\'unconfirmed\':\n                report[\'process_cleanup_unconfirmed\']=True\n            raise\n\n    interpreter = candidate/("Scripts/python.exe" if os.name == "nt" else "bin/python")\n\n    def environment_stage():\n        nonlocal env\n        if hashlib.sha256(lock_text.encode()).hexdigest() != expected["candidate_lock_sha256"]:\n            raise ValueError("candidate_lock_hash")\n        env = candidate_environment(scratch)\n        lock = scratch/"candidate.lock"\n        lock.write_text(lock_text, encoding="utf-8")\n        (output/"candidate.lock").write_text(lock_text, encoding="utf-8")\n        versions = dict(re.findall(r"^([\\w.-]+)==([^\\s\\\\]+)", lock_text, flags=re.MULTILINE))\n        if any(versions.get(n) != v for n,v in PINNED.items()):\n            raise ValueError("candidate_roots_changed")\n        invoke("create-venv", [sys.executable, "-m", "venv", candidate], 60)\n        configuration = (candidate/"pyvenv.cfg").read_text().lower()\n        if "include-system-site-packages = false" not in configuration:\n            raise ValueError("candidate_inherits_base_packages")\n        tools = expected["build_tools"]\n        files = expected["candidate_wheels"]\n        if {p[\'name\']:p[\'version\'] for p in files} != versions:\n            raise ValueError("candidate_wheel_closure")\n        download_bytes = sum(p["bytes"] for p in files)\n        # This stated package-expansion forecast is replaced by exact wheel ZIP sizes below.\n        facts = require_space(scratch, output, 4*download_bytes+BUILD_RESERVE)\n        facts.update(download_bytes=download_bytes, package_expansion_forecast_multiplier=4)\n        save_json(output/"storage-before-wheels.json", facts)\n        save_json(output/"wheel-files.json", files)\n        bootstrap = scratch/"pip-bootstrap.lock"\n        p = tools["pip"]\n        bootstrap.write_text(f"pip=={p[\'version\']} --hash=sha256:{p[\'wheels\'][0][\'sha256\']}\\n")\n        invoke("bootstrap-pip", [interpreter,"-m","pip","--isolated","install","--no-cache-dir",\n               "--no-deps","--require-hashes","--report",output/"pip-bootstrap.json",\n               "--index-url","https://pypi.org/simple","-r",bootstrap], 120)\n        pip = [interpreter,"-m","pip","--isolated","--disable-pip-version-check"]\n        common = ["--no-cache-dir","--only-binary=:all:","--require-hashes","-r",lock]\n        wheels = scratch/"wheels"\n        wheels.mkdir()\n        direct_lock = scratch/"candidate-download.lock"\n        direct_lock.write_text("".join(f"{p[\'name\']} @ {p[\'url\']} --hash=sha256:{p[\'sha256\']}\\n" for p in files))\n        invoke("download-candidate", pip+["download","--no-deps","--dest",wheels,"--no-cache-dir",\n               "--require-hashes","--only-binary=:all:","-r",direct_lock],900)\n        tool_lock = scratch/"build-tools.lock"\n        tool_lock.write_text("".join(f"{n}=={tools[n][\'version\']} --hash=sha256:{tools[n][\'wheels\'][0][\'sha256\']}\\n"\n                                    for n in ("setuptools","wheel")))\n        invoke("download-build-tools", pip+["download","--no-deps","--dest",wheels,"--only-binary=:all:",\n               "--require-hashes","--no-cache-dir","-r",tool_lock,"--index-url","https://pypi.org/simple"],120)\n        expected_files = {p[\'filename\']:p for p in files}\n        expected_files.update({tools[n][\'wheels\'][0][\'filename\']:tools[n][\'wheels\'][0] for n in (\'setuptools\',\'wheel\')})\n        actual_files = {p.name:p for p in wheels.iterdir()}\n        if set(actual_files) != set(expected_files):\n            raise ValueError("wheel_file_set")\n        expanded = 0\n        for name,path in actual_files.items():\n            wanted = expected_files[name]\n            if path.stat().st_size != wanted[\'bytes\'] or sha256_file(path) != wanted[\'sha256\']:\n                raise ValueError(f"wheel_bytes: {name}")\n            with zipfile.ZipFile(path) as archive:\n                expanded += sum(p.file_size for p in archive.infolist())\n        save_json(output/"storage-before-install.json", require_space(scratch,output,expanded+BUILD_RESERVE))\n        invoke("resolve-candidate",pip+["install","--dry-run","--ignore-installed","--no-index",\n               "--find-links",wheels,"--report",output/"resolver.json"]+common,120)\n        resolver = json.loads((output/"resolver.json").read_text())\n        resolved = {p["metadata"]["name"].lower().replace("_","-"): p["metadata"]["version"]\n                    for p in resolver["install"]}\n        if resolved != versions:\n            raise ValueError("actual_resolver_closure_differs_from_candidate")\n        invoke("install-candidate",pip+["install","--no-index","--find-links",wheels,"--no-compile",\n               "--report",output/"install.json"]+common,300)\n        constraints = scratch/"build-constraints.txt"\n        constraints.write_text("".join(f"{n}=={tools[n][\'version\']}\\n" for n in (\'setuptools\',\'wheel\')))\n        built = scratch/"built"\n        built.mkdir()\n        invoke("build-source",pip+["wheel","--no-deps","--no-index","--find-links",wheels,\n               "--build-constraint",constraints,"--wheel-dir",built,source,"-v"],120)\n        built_files = list(built.glob(\'*.whl\'))\n        if len(built_files) != 1:\n            raise ValueError("source_wheel_count")\n        save_json(output/"source-wheel.json", {"filename":built_files[0].name,\n                  "bytes":built_files[0].stat().st_size,"sha256":sha256_file(built_files[0]),\n                  "build_tools":{n:tools[n][\'version\'] for n in (\'setuptools\',\'wheel\')}})\n        invoke("install-source",pip+["install","--no-deps","--no-index","--no-compile",\n               "--report",output/"source-install.json",built_files[0]],60)\n        return {"mode":"clean-venv-offline-wheel-install","versions":versions,\n                "resolved_packages":len(versions),"download_bytes":download_bytes,\n                "wheel_expanded_bytes":expanded,"build_tools":tools}\n\n    def packages_stage():\n        invoke("pip-check", [interpreter,"-m","pip","--isolated","check"],30)\n        probe = scratch/"installed_probe.py"\n        versions = report[\'stages\'][\'environment\'][\'result\'][\'versions\']\n        probe.write_text(installed_probe_script(expected,versions),encoding=\'utf-8\')\n        record = invoke("installed-probe", [interpreter,"-I",probe],90)\n        result = json_stdout(output,record)\n        wanted_hashes = {n:p[\'sha256\'] for n,p in expected[\'members\'].items()\n                         if n.startswith(\'recovery/\') and n.endswith(\'.py\')}\n        if (not isinstance(result,dict) or result.get(\'versions\')!=versions\n                or result.get(\'verified_module_hashes\')!=wanted_hashes\n                or result.get(\'qwen_text_class\')!=\'Qwen3_5ForCausalLM\'\n                or result.get(\'torch_cuda_build\')!=\'12.8\' or result.get(\'model_generations\')!=0):\n            raise ValueError(\'installed_probe_schema_or_provenance\')\n        save_json(output/"installed-provenance.json",result)\n        record = invoke("pip-list", [interpreter,"-m","pip","--isolated","list","--format=json"],30)\n        save_json(output/"installed-packages.json",json_stdout(output,record))\n        freeze = invoke("pip-freeze", [interpreter,"-m","pip","--isolated","freeze","--all"],30)\n        shutil.copyfile(output/freeze[\'stdout\'],output/\'installed-freeze.txt\')\n        return result\n\n    def tests_stage(name,args,count,bound):\n        record = invoke(name,[interpreter]+args,bound,cwd=source,source_mode=True)\n        return test_result(output,record,count)\n\n    result = {"zip_path":None,"zip_sha256":None,"run_directory":str(output)}\n    try:\n        save_json(output/"expected-source-manifest.json",expected)\n        stage("input_inventory",[],input_stage)\n        stage("source",["input_inventory"],source_stage)\n        stage("cpu_host",[],host_stage)\n        stage("command_cleanup",["source","cpu_host"],lambda:process_tree_diagnostic(output,scratch,deadline,candidate_environment(scratch)))\n        stage("environment",["source","cpu_host","command_cleanup"],environment_stage)\n        stage("installed_packages",["environment"],packages_stage)\n        stage("focused_checks",["installed_packages"],lambda:tests_stage("focused-checks",["-m","unittest",*FOCUSED,"-q"],81,360))\n        stage("native_9b",["installed_packages"],lambda:tests_stage("native-9b",["-m","tests.tokenizer_inference","--tokenizer","metadata/9B"],4,60))\n        stage("native_4b",["installed_packages"],lambda:tests_stage("native-4b",["-m","tests.tokenizer_inference","--tokenizer","metadata/4B"],4,60))\n        stage("native_capacity",["installed_packages"],lambda:tests_stage("native-capacity",["-m","tests.tokenizer_capacity","--tokenizer","metadata/9B"],8,90))\n        stage("filesystem_backup",[],lambda:sqlite_diagnostic(output))\n    except KeyboardInterrupt:\n        report[\'root_blocker\'] = report[\'root_blocker\'] or \'user_interruption\'\n        for state in report[\'stages\'].values():\n            if state[\'status\'] in {\'running\',\'not_run\'}:\n                state.update(status=\'blocked\',reason=\'user_interruption\')\n    finally:\n        states = report[\'stages\']\n        passed = all(s[\'status\']==\'passed\' for s in states.values())\n        tests = sum(s.get(\'result\',{}).get(\'tests\',0) for s in states.values() if s[\'status\']==\'passed\')\n        summary = {"preparation_checks_passed":passed,"root_blocker":report[\'root_blocker\'],\n                   "tests_passed":tests,"model_generations":0,"model_smoke_ready":False,\n                   "gpu_native_diagnostics":None,"model_assets_verified":None,"model_smoke":None,\n                   "provider_assignment_utc":None,"provider_release_utc":None,\n                   "observed_cpu_wall_seconds":time.monotonic()-started,\n                   "next_action":("Download the ZIP and stop the session. Return it for candidate-host review before GPU/model work."\n                    if passed else "Download the ZIP and stop the session. Return the root-blocker evidence for review; do not rerun or repair fixtures.")}\n        if report[\'root_blocker\'] and \'payload_missing\' in report[\'root_blocker\']:\n            summary[\'next_action\']=\'Attach the private dataset containing unchanged m5-kaggle-source.payload. Download this ZIP, stop the session, and return the evidence.\'\n        report[\'summary\']=summary\n        save_json(output/\'preflight-report.json\',report)\n        save_json(output/\'summary.json\',summary)\n        try:\n            if scratch.exists():\n                storage=filesystem_facts({\'scratch\':scratch,\'output\':output})\n                storage[\'models\']={n:{\'selected_file_bytes\':b,\'assets_staged\':False,\'quota_admitted\':False}\n                                   for n,b in MODEL_SELECTED_BYTES.items()}\n                save_json(output/\'storage-after.json\',storage)\n        except Exception as exc:\n            report[\'storage_after_error\']=f\'{type(exc).__name__}: {exc}\'\n            save_json(output/\'preflight-report.json\',report)\n        try:\n            result.update(export_evidence(output))\n        except Exception as exc:\n            report[\'export_error\']=f\'{type(exc).__name__}: {exc}\'\n            save_json(output/\'preflight-report.json\',report)\n        result.update(summary=summary,report=report)\n    return result\n'
EXPECTED=json.loads('{"archive_bytes": 109248341, "archive_sha256": "57f99508f47f5571769891e39603db0c2b15c99038e0a186f3e2d40a9d48622f", "build_commit": "42fb405e09d879c1bd4594e9250cfbe53ad3e373", "build_python": "3.11.9", "build_tools": {"pip": {"requires_python": ">=3.10", "version": "26.2.1", "wheels": [{"bytes": 1816632, "filename": "pip-26.2.1-py3-none-any.whl", "sha256": "71138adf1f4ca900cdb7d289c21b7494329f2332b6d85f0e1c42108c0384ed3e", "url": "https://files.pythonhosted.org/packages/f3/6e/1736e5b4ae2b778ef2f81c47d797de9f891d4d8acb047a24ca37a60294dd/pip-26.2.1-py3-none-any.whl"}]}, "setuptools": {"requires_python": ">=3.10", "version": "84.0.0", "wheels": [{"bytes": 818216, "filename": "setuptools-84.0.0-py3-none-any.whl", "sha256": "51a52592b3b99e102b609654876bd65f19f999935166d1352678931132b0c670", "url": "https://files.pythonhosted.org/packages/95/9c/c510029fc6ef33a6275cd2c5d3cecd6613dfd6aa401d57c54f1c18852ccf/setuptools-84.0.0-py3-none-any.whl"}]}, "wheel": {"requires_python": ">=3.9", "version": "0.48.0", "wheels": [{"bytes": 33320, "filename": "wheel-0.48.0-py3-none-any.whl", "sha256": "3217dcc807155e45db462d7ef2431f5ddda0d7273b700d05a67b271ceb1287ab", "url": "https://files.pythonhosted.org/packages/2e/29/69cfbb602cd91690c55d38ba9fe53e6a7e76a6fa647bf38f19c138d25449/wheel-0.48.0-py3-none-any.whl"}]}}, "candidate_lock_sha256": "9499aa04d5cf0faa3843f994071df49365c425bb3a0bb45f6335bdbaaa142ae5", "candidate_wheels": [{"bytes": 389246, "filename": "accelerate-1.14.0-py3-none-any.whl", "name": "accelerate", "sha256": "e94390c2863b873be18f623f9df48a0d8fe5eff13ea7f1a00092b0a7904888c6", "url": "https://files.pythonhosted.org/packages/a8/db/253133d7e7cb40d3af384bb2f5c0b4a2b7fdcffbc95c688cc67a20a3c103/accelerate-1.14.0-py3-none-any.whl", "version": "1.14.0"}, {"bytes": 17405, "filename": "aiosqlite-0.22.1-py3-none-any.whl", "name": "aiosqlite", "sha256": "21c002eb13823fad740196c5a2e9d8e62f6243bd9e7e4a1f87fb5e44ecb4fceb", "url": "https://files.pythonhosted.org/packages/00/b7/e3bf5133d697a08128598c8d0abc5e16377b51465a33756de24fa7dee953/aiosqlite-0.22.1-py3-none-any.whl", "version": "0.22.1"}, {"bytes": 5302, "filename": "annotated_doc-0.0.5-py3-none-any.whl", "name": "annotated-doc", "sha256": "117bac03a25ede5df5440e855b32d556049ca169ead221505badf432fed4b101", "url": "https://files.pythonhosted.org/packages/3e/30/e900b21425a860e195f32e37657aa1f7c7f2b1bfb26f03ca209b90933c06/annotated_doc-0.0.5-py3-none-any.whl", "version": "0.0.5"}, {"bytes": 13427, "filename": "annotated_types-0.8.0-py3-none-any.whl", "name": "annotated-types", "sha256": "f072f4d804ea359e4eaf198b1af7a8b0943881a87f31bb764f8bf219bb9419e0", "url": "https://files.pythonhosted.org/packages/99/91/8acff4f5e50511b911bbccb72b8628a49c68ce14148cd9f6431094859a90/annotated_types-0.8.0-py3-none-any.whl", "version": "0.8.0"}, {"bytes": 132079, "filename": "anyio-4.15.1-py3-none-any.whl", "name": "anyio", "sha256": "6152fdbbf9a77fdec97731721bebf7c4c44f7c29b424b0065826173efc7ed101", "url": "https://files.pythonhosted.org/packages/12/b8/4bd346e22b28902df4d651910f5242c28d84e4a5c2435ca5c3f797ed7e2e/anyio-4.15.1-py3-none-any.whl", "version": "4.15.1"}, {"bytes": 60651714, "filename": "bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl", "name": "bitsandbytes", "sha256": "54b771f06e1a3c73af5c7f16ccf0fc23a846052813d4b008d10cb6e017dd1c8c", "url": "https://files.pythonhosted.org/packages/19/57/3443d6f183436fbdaf5000aac332c4d5ddb056665d459244a5608e98ae92/bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl", "version": "0.49.2"}, {"bytes": 136983, "filename": "certifi-2026.7.22-py3-none-any.whl", "name": "certifi", "sha256": "62f22742b58a1a33014a2b6b706588a8d7e2a88ae7bd1a6ebe8c992928483775", "url": "https://files.pythonhosted.org/packages/0b/a7/71ac2cff56fec219ed242bb11b8efb69fcc4bec75db06fb7bfe35de520e6/certifi-2026.7.22-py3-none-any.whl", "version": "2026.7.22"}, {"bytes": 254660, "filename": "charset_normalizer-3.5.2-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl", "name": "charset-normalizer", "sha256": "7218e8f32b0956cfcd048fd42d9d5779809745ca1d86113ca56f66e7ae1549c4", "url": "https://files.pythonhosted.org/packages/4b/85/41d27f234b82e47c167a5f6c0f62501dc0c640585ff4aba79e08a390336a/charset_normalizer-3.5.2-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl", "version": "3.5.2"}, {"bytes": 125251, "filename": "click-8.5.0-py3-none-any.whl", "name": "click", "sha256": "255bc9599cf7748b4b1a446ccc735421bd08a2ae529a8b88597d3de5664ee360", "url": "https://files.pythonhosted.org/packages/58/50/6c0d534c5f134586a8e1ba4e330569e32f057e33372ae556463212fb4cd3/click-8.5.0-py3-none-any.whl", "version": "8.5.0"}, {"bytes": 7398149, "filename": "cuda_bindings-12.9.9-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl", "name": "cuda-bindings", "sha256": "d399447bf5d929d6f59ec4e2c7b26d11ef21e8be5e88995e41c705db1a7c1363", "url": "https://files.pythonhosted.org/packages/ef/c8/0d7703abc9df0871965f9b6eafd881a261f156a168beca9050052ca3e4ab/cuda_bindings-12.9.9-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl", "version": "12.9.9"}, {"bytes": 62561, "filename": "cuda_pathfinder-1.8.3-py3-none-any.whl", "name": "cuda-pathfinder", "sha256": "e29e59829c297a7a5233bd9cc71094fc5bddbd076951482670178f9eade39b1f", "url": "https://files.pythonhosted.org/packages/b9/fb/f8e1890428f9f590b4beebd63b068aac1ce32a3331510c847b9f9a78f261/cuda_pathfinder-1.8.3-py3-none-any.whl", "version": "1.8.3"}, {"bytes": 2283, "filename": "cuda_toolkit-12.8.1-py2.py3-none-any.whl", "name": "cuda-toolkit", "sha256": "adc7906af4ecbf9a352f9dca5734eceb21daec281ccfcf5675e1d2f724fc2cba", "url": "https://files.pythonhosted.org/packages/d4/c8/7dce3a0b15b42a3b58e7d96eb22a687d3bf2c44e01d149a6874629cd9938/cuda_toolkit-12.8.1-py2.py3-none-any.whl", "version": "12.8.1"}, {"bytes": 20277, "filename": "distro-1.9.0-py3-none-any.whl", "name": "distro", "sha256": "7bffd925d65168f85027d8da9af6bddab658135b840670a223589bc0c8ef02b2", "url": "https://files.pythonhosted.org/packages/12/b3/231ffd4ab1fc9d679809f356cebee130ac7daa00d6d6f3206dd4fd137e9e/distro-1.9.0-py3-none-any.whl", "version": "1.9.0"}, {"bytes": 111732, "filename": "filelock-4.0.12-py3-none-any.whl", "name": "filelock", "sha256": "5f17ee83ecee8a6f3e389c75822fb70a1c2f0506b99438a6dffa1d56793588c8", "url": "https://files.pythonhosted.org/packages/da/a1/b047e1c486a2c66283633b7dbb73872ca16ca1ce6a426f70d4d037fc8e1c/filelock-4.0.12-py3-none-any.whl", "version": "4.0.12"}, {"bytes": 221738, "filename": "fsspec-2026.9.0-py3-none-any.whl", "name": "fsspec", "sha256": "8dd6e646e99ea382bd85f97a45e6b526a442d79423a7dc673f1e2756d05fcb5f", "url": "https://files.pythonhosted.org/packages/6c/c0/a98505f18594f1bce828bb159cec0fcf9860562f1a2c85913409fc8f3d9e/fsspec-2026.9.0-py3-none-any.whl", "version": "2026.9.0"}, {"bytes": 37515, "filename": "h11-0.16.0-py3-none-any.whl", "name": "h11", "sha256": "63cf8bbe7522de3bf65932fda1d9c2772064ffb3dae62d55932da54b31cb6c86", "url": "https://files.pythonhosted.org/packages/04/4b/29cac41a4d98d144bf5f6d33995617b185d14b22401f75ca86f384e87ff1/h11-0.16.0-py3-none-any.whl", "version": "0.16.0"}, {"bytes": 4212180, "filename": "hf_xet-1.7.0-cp38-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl", "name": "hf-xet", "sha256": "2814a6e999d13464c4d679b788cc5d784eb5a4edfc638a31f10e9a11ab531ef8", "url": "https://files.pythonhosted.org/packages/a3/4c/2bf3b66c215d409655f28de1622393dde04c9461280d48c7924bb3b2decd/hf_xet-1.7.0-cp38-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl", "version": "1.7.0"}, {"bytes": 78784, "filename": "httpcore-1.0.9-py3-none-any.whl", "name": "httpcore", "sha256": "2d400746a40668fc9dec9810239072b40b4484b640a8c38fd654a024c7a1bf55", "url": "https://files.pythonhosted.org/packages/7e/f5/f66802a942d491edb555dd61e3a9961140fd64c90bce1eafd741609d334d/httpcore-1.0.9-py3-none-any.whl", "version": "1.0.9"}, {"bytes": 83423, "filename": "httpcore2-2.13.1-py3-none-any.whl", "name": "httpcore2", "sha256": "e1e05d4f25f7d7d496bfb96748f6f4b67657b03da069b3a68c36069f3db73d0a", "url": "https://files.pythonhosted.org/packages/09/ba/a4568248771ce81957bfb7cc600264a40fbcda092391ee1c415c50be4bea/httpcore2-2.13.1-py3-none-any.whl", "version": "2.13.1"}, {"bytes": 73517, "filename": "httpx-0.28.1-py3-none-any.whl", "name": "httpx", "sha256": "d909fcccc110f8c7faf814ca82a9a4d816bc5a6dbfea25d6591d6985b8ba59ad", "url": "https://files.pythonhosted.org/packages/2a/39/e50c7c3a983047577ee07d2a9e53faf5a69493943ec3f6a384bdc792deb2/httpx-0.28.1-py3-none-any.whl", "version": "0.28.1"}, {"bytes": 95597, "filename": "httpx2-2.13.1-py3-none-any.whl", "name": "httpx2", "sha256": "6dff50fabc270ee5fd25d845d0b078ed20564579744d6d962850975996d2f9a4", "url": "https://files.pythonhosted.org/packages/d8/9c/6fe8931fd9f381042a9e4c7d5a7b4cbf7016b252bec0c99a49fce42c3326/httpx2-2.13.1-py3-none-any.whl", "version": "2.13.1"}, {"bytes": 693398, "filename": "huggingface_hub-1.19.0-py3-none-any.whl", "name": "huggingface-hub", "sha256": "1dc72e1f6b4d6df6b30eb72e57d00514ef453d660f04af2b87f0e67267f31ee0", "url": "https://files.pythonhosted.org/packages/b2/a5/558da89f66464d8d0229ff497e8b8666977de2d8cf48c28a2862ecf1250f/huggingface_hub-1.19.0-py3-none-any.whl", "version": "1.19.0"}, {"bytes": 69583, "filename": "idna-3.20-py3-none-any.whl", "name": "idna", "sha256": "ab7ae7122974553370f0bdb919e1a960b2cd1bc1ef0276416d896db81c14582c", "url": "https://files.pythonhosted.org/packages/58/a2/bb081bab032533a855d44de1d56f8e8426114ff1ba5d1f07a438a0a654f8/idna-3.20-py3-none-any.whl", "version": "3.20"}, {"bytes": 134899, "filename": "jinja2-3.1.6-py3-none-any.whl", "name": "jinja2", "sha256": "85ece4451f492d0c13c5dd7c13a64681a86afae63a5f347908daf103ce6d2f67", "url": "https://files.pythonhosted.org/packages/62/a1/3d680cbfd5f4b8f15abc1d571870c5fc3e594bb582bc3b64ea099db13e56/jinja2-3.1.6-py3-none-any.whl", "version": "3.1.6"}, {"bytes": 12898, "filename": "jsonpatch-1.33-py2.py3-none-any.whl", "name": "jsonpatch", "sha256": "0ae28c0cd062bbd8b8ecc26d7d164fbbea9652a1a3693f3b956c1eae5145dade", "url": "https://files.pythonhosted.org/packages/73/07/02e16ed01e04a374e644b575638ec7987ae846d25ad97bcc9945a3ee4b0e/jsonpatch-1.33-py2.py3-none-any.whl", "version": "1.33"}, {"bytes": 8236, "filename": "jsonpointer-3.2.0-py3-none-any.whl", "name": "jsonpointer", "sha256": "3e25c9bcadf5ae20c3420922234eaba1ae36ab9bef64cd72454aeaceb0b29aad", "url": "https://files.pythonhosted.org/packages/b6/b7/5b934a4fa2ea1e28177fb74a08de7aff8ee160fb6e5ddc6574c7af1a808d/jsonpointer-3.2.0-py3-none-any.whl", "version": "3.2.0"}, {"bytes": 572080, "filename": "langchain_core-1.6.6-py3-none-any.whl", "name": "langchain-core", "sha256": "d9a71e312ab7443701238750e04a57aefc037c4f1ec82837c8a1609d9a14a9b1", "url": "https://files.pythonhosted.org/packages/90/10/8c00b4dc094886314c907ec7ead258dfee28948cf44175976cc4fba9aba8/langchain_core-1.6.6-py3-none-any.whl", "version": "1.6.6"}, {"bytes": 7327, "filename": "langchain_protocol-0.0.19-py3-none-any.whl", "name": "langchain-protocol", "sha256": "4cdf879a492a35980fd859ae792d3c65458ccaae504e183c9a10d7eac1f0720f", "url": "https://files.pythonhosted.org/packages/80/c9/f6cbf357d48ccbd18bb394433b1fd7ad9be004eed9377ad08bb85777e5e6/langchain_protocol-0.0.19-py3-none-any.whl", "version": "0.0.19"}, {"bytes": 250206, "filename": "langgraph-1.2.12-py3-none-any.whl", "name": "langgraph", "sha256": "95403af7b510de8d79164742f71daaf866c69ca804a8cb72bef2cc1fa1ab9813", "url": "https://files.pythonhosted.org/packages/52/27/5a02054c535ed6fcfb2ab1e5c33c51c810cf1e773194ad28c928f0473509/langgraph-1.2.12-py3-none-any.whl", "version": "1.2.12"}, {"bytes": 56833, "filename": "langgraph_checkpoint-4.2.0-py3-none-any.whl", "name": "langgraph-checkpoint", "sha256": "0547fd228935a0b758865de3a3d6d7a2537c308895d0f9ab092ce9151b5da942", "url": "https://files.pythonhosted.org/packages/05/71/3b475f09bd57d3a5649792c66353312b4432afd843f301739dfcebd157f0/langgraph_checkpoint-4.2.0-py3-none-any.whl", "version": "4.2.0"}, {"bytes": 40785, "filename": "langgraph_checkpoint_sqlite-3.1.1-py3-none-any.whl", "name": "langgraph-checkpoint-sqlite", "sha256": "8505c54c94a658080525d7e6780fdd4e0c078ff2566b30d399c02cc9f9af1c63", "url": "https://files.pythonhosted.org/packages/f5/b9/e458601a1718337839bcfeec9d1b27b8b16ce135be2bd50ed0395d33a878/langgraph_checkpoint_sqlite-3.1.1-py3-none-any.whl", "version": "3.1.1"}, {"bytes": 41043, "filename": "langgraph_prebuilt-1.1.0-py3-none-any.whl", "name": "langgraph-prebuilt", "sha256": "51e311747d755b751d5c6b39b0c1446124d3a7643d2515017e6714b323508fc9", "url": "https://files.pythonhosted.org/packages/e9/43/3fe1a700b8490ed02679cdbbc8c915eb23a092faf496c9c1118abcd10be3/langgraph_prebuilt-1.1.0-py3-none-any.whl", "version": "1.1.0"}, {"bytes": 162390, "filename": "langgraph_sdk-0.4.6-py3-none-any.whl", "name": "langgraph-sdk", "sha256": "57122b02bda41d37a97a5e09eb018aa8c9b3dc85c433880073779bd9324b7eea", "url": "https://files.pythonhosted.org/packages/6c/c7/69d13026db3577c157877dea47a9dc5fffd3dcde2d1ab8009fa8f7e5f6a6/langgraph_sdk-0.4.6-py3-none-any.whl", "version": "0.4.6"}, {"bytes": 813097, "filename": "langsmith-0.14.4-py3-none-any.whl", "name": "langsmith", "sha256": "ae0d98c2b0f9292ef7e12451d480937d79a0e7dcf2412ba847de6f053391b44a", "url": "https://files.pythonhosted.org/packages/e4/1f/5c2919fef6e3cf15167699abeb069a371d68da14a0e5cab89eeea11261d7/langsmith-0.14.4-py3-none-any.whl", "version": "0.14.4"}, {"bytes": 91687, "filename": "markdown_it_py-4.2.0-py3-none-any.whl", "name": "markdown-it-py", "sha256": "9f7ebbcd14fe59494226453aed97c1070d83f8d24b6fc3a3bcf9a38092641c4a", "url": "https://files.pythonhosted.org/packages/b3/81/4da04ced5a082363ecfa159c010d200ecbd959ae410c10c0264a38cac0f5/markdown_it_py-4.2.0-py3-none-any.whl", "version": "4.2.0"}, {"bytes": 23017, "filename": "markupsafe-3.0.4-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl", "name": "markupsafe", "sha256": "434139499bb20b502ed3baa1f169e618f924a97e7a777fea1a49446d80106cf6", "url": "https://files.pythonhosted.org/packages/63/e0/cec6865dfe88cb48fedd4b20aed6af5158e41092adcbf3e028bcc6ec2108/markupsafe-3.0.4-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl", "version": "3.0.4"}, {"bytes": 9979, "filename": "mdurl-0.1.2-py3-none-any.whl", "name": "mdurl", "sha256": "84008a41e51615a49fc9966191ff91509e3c40b939176e643fd50a5c2196b8f8", "url": "https://files.pythonhosted.org/packages/b3/38/89ba8ad64ae25be8de66a6d463314cf1eb366222074cfda9ee839c56a4b4/mdurl-0.1.2-py3-none-any.whl", "version": "0.1.2"}, {"bytes": 536198, "filename": "mpmath-1.3.0-py3-none-any.whl", "name": "mpmath", "sha256": "a0b2b9fe80bbcd81a6647ff13108738cfb482d481d826cc0e02f5b35e5c88d2c", "url": "https://files.pythonhosted.org/packages/43/e3/7d92a15f894aa0c9c4b49b8ee9ac9850d6e63b03c9c32c0367a13ae62209/mpmath-1.3.0-py3-none-any.whl", "version": "1.3.0"}, {"bytes": 2142205, "filename": "networkx-3.7-py3-none-any.whl", "name": "networkx", "sha256": "e3fd2c13a7814cee3746340d8d7f8598a67f16a58bf47fb7f8793fab6efca1b0", "url": "https://files.pythonhosted.org/packages/7e/cd/fe58041e9011f307c490e3e17dd48cc516448f7c698a3f2d9d9d65d7e6a8/networkx-3.7-py3-none-any.whl", "version": "3.7"}, {"bytes": 16708577, "filename": "numpy-2.5.3-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl", "name": "numpy", "sha256": "a5fa86b80fd24bcd1aff83ad23be44ea323de3f787be8f8b15d4a65621e25321", "url": "https://files.pythonhosted.org/packages/3a/1b/3b16a9bc514a440a7a0883684111dcb1ef1aee960af2ca95da8fc775f124/numpy-2.5.3-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl", "version": "2.5.3"}, {"bytes": 594346921, "filename": "nvidia_cublas_cu12-12.8.4.1-py3-none-manylinux_2_27_x86_64.whl", "name": "nvidia-cublas-cu12", "sha256": "8ac4e771d5a348c551b2a426eda6193c19aa630236b418086020df5ba9667142", "url": "https://files.pythonhosted.org/packages/dc/61/e24b560ab2e2eaeb3c839129175fb330dfcfc29e5203196e5541a4c44682/nvidia_cublas_cu12-12.8.4.1-py3-none-manylinux_2_27_x86_64.whl", "version": "12.8.4.1"}, {"bytes": 10248621, "filename": "nvidia_cuda_cupti_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl", "name": "nvidia-cuda-cupti-cu12", "sha256": "ea0cb07ebda26bb9b29ba82cda34849e73c166c18162d3913575b0c9db9a6182", "url": "https://files.pythonhosted.org/packages/f8/02/2adcaa145158bf1a8295d83591d22e4103dbfd821bcaf6f3f53151ca4ffa/nvidia_cuda_cupti_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl", "version": "12.8.90"}, {"bytes": 88040029, "filename": "nvidia_cuda_nvrtc_cu12-12.8.93-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl", "name": "nvidia-cuda-nvrtc-cu12", "sha256": "a7756528852ef889772a84c6cd89d41dfa74667e24cca16bb31f8f061e3e9994", "url": "https://files.pythonhosted.org/packages/05/6b/32f747947df2da6994e999492ab306a903659555dddc0fbdeb9d71f75e52/nvidia_cuda_nvrtc_cu12-12.8.93-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl", "version": "12.8.93"}, {"bytes": 954765, "filename": "nvidia_cuda_runtime_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl", "name": "nvidia-cuda-runtime-cu12", "sha256": "adade8dcbd0edf427b7204d480d6066d33902cab2a4707dcfc48a2d0fd44ab90", "url": "https://files.pythonhosted.org/packages/0d/9b/a997b638fcd068ad6e4d53b8551a7d30fe8b404d6f1804abf1df69838932/nvidia_cuda_runtime_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl", "version": "12.8.90"}, {"bytes": 657906812, "filename": "nvidia_cudnn_cu12-9.19.0.56-py3-none-manylinux_2_27_x86_64.whl", "name": "nvidia-cudnn-cu12", "sha256": "ac6ad90a075bb33a94f2b4cf4622eac13dd4dc65cf6dd9c7572a318516a36625", "url": "https://files.pythonhosted.org/packages/c5/41/65225d42fba06fb3dd3972485ea258e7dd07a40d6e01c95da6766ad87354/nvidia_cudnn_cu12-9.19.0.56-py3-none-manylinux_2_27_x86_64.whl", "version": "9.19.0.56"}, {"bytes": 193118695, "filename": "nvidia_cufft_cu12-11.3.3.83-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl", "name": "nvidia-cufft-cu12", "sha256": "4d2dd21ec0b88cf61b62e6b43564355e5222e4a3fb394cac0db101f2dd0d4f74", "url": "https://files.pythonhosted.org/packages/1f/13/ee4e00f30e676b66ae65b4f08cb5bcbb8392c03f54f2d5413ea99a5d1c80/nvidia_cufft_cu12-11.3.3.83-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl", "version": "11.3.3.83"}, {"bytes": 1197834, "filename": "nvidia_cufile_cu12-1.13.1.3-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl", "name": "nvidia-cufile-cu12", "sha256": "1d069003be650e131b21c932ec3d8969c1715379251f8d23a1860554b1cb24fc", "url": "https://files.pythonhosted.org/packages/bb/fe/1bcba1dfbfb8d01be8d93f07bfc502c93fa23afa6fd5ab3fc7c1df71038a/nvidia_cufile_cu12-1.13.1.3-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl", "version": "1.13.1.3"}, {"bytes": 63619976, "filename": "nvidia_curand_cu12-10.3.9.90-py3-none-manylinux_2_27_x86_64.whl", "name": "nvidia-curand-cu12", "sha256": "b32331d4f4df5d6eefa0554c565b626c7216f87a06a4f56fab27c3b68a830ec9", "url": "https://files.pythonhosted.org/packages/fb/aa/6584b56dc84ebe9cf93226a5cde4d99080c8e90ab40f0c27bda7a0f29aa1/nvidia_curand_cu12-10.3.9.90-py3-none-manylinux_2_27_x86_64.whl", "version": "10.3.9.90"}, {"bytes": 267506905, "filename": "nvidia_cusolver_cu12-11.7.3.90-py3-none-manylinux_2_27_x86_64.whl", "name": "nvidia-cusolver-cu12", "sha256": "4376c11ad263152bd50ea295c05370360776f8c3427b30991df774f9fb26c450", "url": "https://files.pythonhosted.org/packages/85/48/9a13d2975803e8cf2777d5ed57b87a0b6ca2cc795f9a4f59796a910bfb80/nvidia_cusolver_cu12-11.7.3.90-py3-none-manylinux_2_27_x86_64.whl", "version": "11.7.3.90"}, {"bytes": 288216466, "filename": "nvidia_cusparse_cu12-12.5.8.93-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl", "name": "nvidia-cusparse-cu12", "sha256": "1ec05d76bbbd8b61b06a80e1eaf8cf4959c3d4ce8e711b65ebd0443bb0ebb13b", "url": "https://files.pythonhosted.org/packages/c2/f5/e1854cb2f2bcd4280c44736c93550cc300ff4b8c95ebe370d0aa7d2b473d/nvidia_cusparse_cu12-12.5.8.93-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl", "version": "12.5.8.93"}, {"bytes": 287193691, "filename": "nvidia_cusparselt_cu12-0.7.1-py3-none-manylinux2014_x86_64.whl", "name": "nvidia-cusparselt-cu12", "sha256": "f1bb701d6b930d5a7cea44c19ceb973311500847f81b634d802b7b539dc55623", "url": "https://files.pythonhosted.org/packages/56/79/12978b96bd44274fe38b5dde5cfb660b1d114f70a65ef962bcbbed99b549/nvidia_cusparselt_cu12-0.7.1-py3-none-manylinux2014_x86_64.whl", "version": "0.7.1"}, {"bytes": 296782137, "filename": "nvidia_nccl_cu12-2.28.9-py3-none-manylinux_2_18_x86_64.whl", "name": "nvidia-nccl-cu12", "sha256": "485776daa8447da5da39681af455aa3b2c2586ddcf4af8772495e7c532c7e5ab", "url": "https://files.pythonhosted.org/packages/4a/4e/44dbb46b3d1b0ec61afda8e84837870f2f9ace33c564317d59b70bc19d3e/nvidia_nccl_cu12-2.28.9-py3-none-manylinux_2_18_x86_64.whl", "version": "2.28.9"}, {"bytes": 39254836, "filename": "nvidia_nvjitlink_cu12-12.8.93-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl", "name": "nvidia-nvjitlink-cu12", "sha256": "81ff63371a7ebd6e6451970684f916be2eab07321b73c9d244dc2b4da7f73b88", "url": "https://files.pythonhosted.org/packages/f6/74/86a07f1d0f42998ca31312f998bd3b9a7eff7f52378f4f270c8679c77fb9/nvidia_nvjitlink_cu12-12.8.93-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl", "version": "12.8.93"}, {"bytes": 139103095, "filename": "nvidia_nvshmem_cu12-3.4.5-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl", "name": "nvidia-nvshmem-cu12", "sha256": "042f2500f24c021db8a06c5eec2539027d57460e1c1a762055a6554f72c369bd", "url": "https://files.pythonhosted.org/packages/b5/09/6ea3ea725f82e1e76684f0708bbedd871fc96da89945adeba65c3835a64c/nvidia_nvshmem_cu12-3.4.5-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl", "version": "3.4.5"}, {"bytes": 89954, "filename": "nvidia_nvtx_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl", "name": "nvidia-nvtx-cu12", "sha256": "5b17e2001cc0d751a5bc2c6ec6d26ad95913324a4adb86788c944f8ce9ba441f", "url": "https://files.pythonhosted.org/packages/a2/eb/86626c1bbc2edb86323022371c39aa48df6fd8b0a1647bc274577f72e90b/nvidia_nvtx_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl", "version": "12.8.90"}, {"bytes": 130370, "filename": "orjson-3.13.0-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl", "name": "orjson", "sha256": "cbed5f4c4b88d94bcc36115f4c3bb3aa25da1563a5c3328aa3acebce2b083040", "url": "https://files.pythonhosted.org/packages/45/46/f8d83189ff5b7b2ff225a58c5908618cc4e86afe09e65d17a30ac68c9da4/orjson-3.13.0-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl", "version": "3.13.0"}, {"bytes": 212592, "filename": "ormsgpack-1.12.2-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl", "name": "ormsgpack", "sha256": "39c1bd2092880e413902910388be8715f70b9f15f20779d44e673033a6146f2d", "url": "https://files.pythonhosted.org/packages/1c/a2/072343e1413d9443e5a252a8eb591c2d5b1bffbe5e7bfc78c069361b92eb/ormsgpack-1.12.2-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl", "version": "1.12.2"}, {"bytes": 129956, "filename": "packaging-26.3-py3-none-any.whl", "name": "packaging", "sha256": "d7193f7c8e4e93f444fde0262bf90af30e16fa0ad0ad44cb553c87339b23cd1c", "url": "https://files.pythonhosted.org/packages/63/34/ba1c580383c9eada3711951fef0795c80b829a078d72188184bcab9dd527/packaging-26.3-py3-none-any.whl", "version": "26.3"}, {"bytes": 155560, "filename": "psutil-7.2.2-cp36-abi3-manylinux2010_x86_64.manylinux_2_12_x86_64.manylinux_2_28_x86_64.whl", "name": "psutil", "sha256": "076a2d2f923fd4821644f5ba89f059523da90dc9014e85f8e45a5774ca5bc6f9", "url": "https://files.pythonhosted.org/packages/b5/70/5d8df3b09e25bce090399cf48e452d25c935ab72dad19406c77f4e828045/psutil-7.2.2-cp36-abi3-manylinux2010_x86_64.manylinux_2_12_x86_64.manylinux_2_28_x86_64.whl", "version": "7.2.2"}, {"bytes": 472589, "filename": "pydantic-2.13.5-py3-none-any.whl", "name": "pydantic", "sha256": "346a034f080da3755d8e9cb5e00e8b07de1d39e4f6e2c87d8ab7cafa0b269a73", "url": "https://files.pythonhosted.org/packages/eb/47/c95ffc2009878c7aac0c5e08528022dcb885933252a88b5f170058014464/pydantic-2.13.5-py3-none-any.whl", "version": "2.13.5"}, {"bytes": 2066633, "filename": "pydantic_core-2.46.5-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl", "name": "pydantic-core", "sha256": "6f7b393a8b3da82f5c1fc0751e6d01ac6c55b93c18226a60bdfba4a724efafd1", "url": "https://files.pythonhosted.org/packages/d3/f2/9e4de77a6271e07a76d2d58b11c091a979c191ed2939bf80067568b369d2/pydantic_core-2.46.5-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl", "version": "2.46.5"}, {"bytes": 1250147, "filename": "pygments-2.21.0-py3-none-any.whl", "name": "pygments", "sha256": "2363c69b61c4a97c838da3b130dcd6468f4848992b21a82f2a63ec34377137d9", "url": "https://files.pythonhosted.org/packages/71/46/17f022dd3e953bf20a04a028a21ec746d942f8d2af30fa0f124fa0e6a684/pygments-2.21.0-py3-none-any.whl", "version": "2.21.0"}, {"bytes": 801626, "filename": "pyyaml-6.0.3-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl", "name": "pyyaml", "sha256": "0f29edc409a6392443abf94b9cf89ce99889a1dd5376d94316ae5145dfedd5d6", "url": "https://files.pythonhosted.org/packages/74/27/e5b8f34d02d9995b80abcef563ea1f8b56d20134d8f4e5e81733b1feceb2/pyyaml-6.0.3-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl", "version": "6.0.3"}, {"bytes": 807902, "filename": "regex-2026.9.29-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl", "name": "regex", "sha256": "d49c18f1ea294cf4adde2e5ac256e98c82ea9d708462ce4bf799dffa7cfe8a2c", "url": "https://files.pythonhosted.org/packages/9a/8e/c1ed81f55f992f6aa0b699a592a50c1ce9e6d44ff1aee2c14c0537dcef9c/regex-2026.9.29-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl", "version": "2026.9.29"}, {"bytes": 73075, "filename": "requests-2.34.2-py3-none-any.whl", "name": "requests", "sha256": "2a0d60c172f83ac6ab31e4554906c0f3b3588d37b5cb939b1c061f4907e278e0", "url": "https://files.pythonhosted.org/packages/a0/f4/c67b0b3f1b9245e8d266f0f112c500d50e5b4e83cb6f3b71b6528104182a/requests-2.34.2-py3-none-any.whl", "version": "2.34.2"}, {"bytes": 54481, "filename": "requests_toolbelt-1.0.0-py2.py3-none-any.whl", "name": "requests-toolbelt", "sha256": "cccfdd665f0a24fcf4726e690f65639d272bb0637b9b92dfd91a5568ccf6bd06", "url": "https://files.pythonhosted.org/packages/3f/51/d4db610ef29373b879047326cbf6fa98b6c1969d6f6dc423279de2b1be2c/requests_toolbelt-1.0.0-py2.py3-none-any.whl", "version": "1.0.0"}, {"bytes": 310654, "filename": "rich-15.0.0-py3-none-any.whl", "name": "rich", "sha256": "33bd4ef74232fb73fe9279a257718407f169c09b78a87ad3d296f548e27de0bb", "url": "https://files.pythonhosted.org/packages/82/3b/64d4899d73f91ba49a8c18a8ff3f0ea8f1c1d75481760df8c68ef5235bf5/rich-15.0.0-py3-none-any.whl", "version": "15.0.0"}, {"bytes": 516040, "filename": "safetensors-0.8.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl", "name": "safetensors", "sha256": "fd6f3f93c9a0a7cc2788ee63fb763353d4bd2e89b0751bc78fcf7dda00bea774", "url": "https://files.pythonhosted.org/packages/28/50/f203ff3a3ddfe19308efc83c5a3a29ed02bf786732ec35e68bf9162f3365/safetensors-0.8.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl", "version": "0.8.0"}, {"bytes": 1062021, "filename": "setuptools-81.0.0-py3-none-any.whl", "name": "setuptools", "sha256": "fdd925d5c5d9f62e4b74b30d6dd7828ce236fd6ed998a08d81de62ce5a6310d6", "url": "https://files.pythonhosted.org/packages/e1/e3/c164c88b2e5ce7b24d667b9bd83589cf4f3520d97cad01534cd3c4f55fdb/setuptools-81.0.0-py3-none-any.whl", "version": "81.0.0"}, {"bytes": 9755, "filename": "shellingham-1.5.4-py2.py3-none-any.whl", "name": "shellingham", "sha256": "7ecfff8f2fd72616f7481040475a65b2bf8af90a56c89140852d1120324e8686", "url": "https://files.pythonhosted.org/packages/e0/f9/0595336914c5619e5f28a1fb793285925a8cd4b432c9da0a987836c7f822/shellingham-1.5.4-py2.py3-none-any.whl", "version": "1.5.4"}, {"bytes": 10235, "filename": "sniffio-1.3.1-py3-none-any.whl", "name": "sniffio", "sha256": "2f6da418d1f1e0fddd844478f41680e794e6051915791a034ff65e5f100525a2", "url": "https://files.pythonhosted.org/packages/e9/44/75a9c9421471a6c4805dbf2356f7c181a29c1879239abab1ea2cc8f38b40/sniffio-1.3.1-py3-none-any.whl", "version": "1.3.1"}, {"bytes": 163388, "filename": "sqlite_vec-0.1.9-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.manylinux1_x86_64.whl", "name": "sqlite-vec", "sha256": "1515727990b49e79bcaf75fdee2ffc7d461f8b66905013231251f1c8938e7786", "url": "https://files.pythonhosted.org/packages/6f/ad/6afd073b0f817b3e03f9e37ad626ae341805891f23c74b5292818f49ac63/sqlite_vec-0.1.9-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.manylinux1_x86_64.whl", "version": "0.1.9"}, {"bytes": 6299353, "filename": "sympy-1.14.0-py3-none-any.whl", "name": "sympy", "sha256": "e091cc3e99d2141a0ba2847328f5479b05d94a6635cb96148ccb3f34671bd8f5", "url": "https://files.pythonhosted.org/packages/a2/09/77d55d46fd61b4a135c444fc97158ef34a095e5681d0a6c10b75bf356191/sympy-1.14.0-py3-none-any.whl", "version": "1.14.0"}, {"bytes": 32310, "filename": "tenacity-9.2.1-py3-none-any.whl", "name": "tenacity", "sha256": "9e56f17539296baab7beabb08b92f6ee3d7be92d8be72d763360677c2ad6580e", "url": "https://files.pythonhosted.org/packages/d6/26/1ff2b0721ac66a3ec5b1402b333110b352ab0a8724052ac279a7b82d40c4/tenacity-9.2.1-py3-none-any.whl", "version": "9.2.1"}, {"bytes": 3274982, "filename": "tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl", "name": "tokenizers", "sha256": "369cc9fc8cc10cb24143873a0d95438bb8ee257bb80c71989e3ee290e8d72c67", "url": "https://files.pythonhosted.org/packages/2e/76/932be4b50ef6ccedf9d3c6639b056a967a86258c6d9200643f01269211ca/tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl", "version": "0.22.2"}, {"bytes": 820325684, "filename": "torch-2.11.0+cu128-cp313-cp313-manylinux_2_28_x86_64.whl", "name": "torch", "sha256": "db964b33c55035a72ab3e2162287af8f1cc276039c65d015740cc88c26dcedf7", "url": "https://download-r2.pytorch.org/whl/cu128/torch-2.11.0%2Bcu128-cp313-cp313-manylinux_2_28_x86_64.whl", "version": "2.11.0+cu128"}, {"bytes": 80199, "filename": "tqdm-4.70.1-py3-none-any.whl", "name": "tqdm", "sha256": "c293e525e6fef9c20e8728fd4612df02a0aa31bb5fe91ecd93e123b1b7bffa73", "url": "https://files.pythonhosted.org/packages/a7/03/921a3d3c75785aca9ebfbfcabfbc3a1be12e2ab5265deb026d55a5a3f83e/tqdm-4.70.1-py3-none-any.whl", "version": "4.70.1"}, {"bytes": 11150246, "filename": "transformers-5.12.0-py3-none-any.whl", "name": "transformers", "sha256": "500be9eb644ede81c3103eee7687fc36d05dd75d1c76686c3820b26396fe7c7c", "url": "https://files.pythonhosted.org/packages/ff/1f/d385913c38e900d23b728a4188fee625f2fccb306aeebc59be9d91404a5a/transformers-5.12.0-py3-none-any.whl", "version": "5.12.0"}, {"bytes": 188277408, "filename": "triton-3.6.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl", "name": "triton", "sha256": "0075039ff27765480083b1a109999bf27110f3542f1f9fad95f0f9065a36da79", "url": "https://download-r2.pytorch.org/whl/triton-3.6.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl", "version": "3.6.0"}, {"bytes": 18660, "filename": "truststore-0.10.4-py3-none-any.whl", "name": "truststore", "sha256": "adaeaecf1cbb5f4de3b1959b42d41f6fab57b2b1666adb59e89cb0b53361d981", "url": "https://files.pythonhosted.org/packages/19/97/56608b2249fe206a67cd573bc93cd9896e1efb9e98bce9c163bcdc704b88/truststore-0.10.4-py3-none-any.whl", "version": "0.10.4"}, {"bytes": 58409, "filename": "typer-0.25.1-py3-none-any.whl", "name": "typer", "sha256": "75caa44ed46a03fb2dab8808753ffacdbfea88495e74c85a28c5eefcf5f39c89", "url": "https://files.pythonhosted.org/packages/3f/f9/2b3ff4e56e5fa7debfaf9eb135d0da96f3e9a1d5b27222223c7296336e5f/typer-0.25.1-py3-none-any.whl", "version": "0.25.1"}, {"bytes": 45571, "filename": "typing_extensions-4.16.0-py3-none-any.whl", "name": "typing-extensions", "sha256": "481caa481374e813c1b176ada14e97f1f67a4539ce9cfeb3f350d78d6370c2e8", "url": "https://files.pythonhosted.org/packages/49/d3/b8441a820a491ddfc024b0b0cf0393375b75ea13866d9c66727e54c2fc80/typing_extensions-4.16.0-py3-none-any.whl", "version": "4.16.0"}, {"bytes": 14750, "filename": "typing_inspection-0.4.4-py3-none-any.whl", "name": "typing-inspection", "sha256": "65b8397ba37ccbce054456aaccddfc91e6e3083c92824df348d96ca832f3f147", "url": "https://files.pythonhosted.org/packages/67/81/4add07e5172b7ac40d8ed5ff580409a7801a4fe26d529bdd915401dabfbe/typing_inspection-0.4.4-py3-none-any.whl", "version": "0.4.4"}, {"bytes": 135717, "filename": "urllib3-2.8.0-py3-none-any.whl", "name": "urllib3", "sha256": "0cf3cae568d36aa9576b28dfb35f11328f1cb974ca7647d9475ebb86c75ac6e3", "url": "https://files.pythonhosted.org/packages/92/9d/c4e665119135114480843e7ab388fa94d8480650450e6f8e26b70d323a4c/urllib3-2.8.0-py3-none-any.whl", "version": "2.8.0"}, {"bytes": 323874, "filename": "uuid_utils-0.17.1-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl", "name": "uuid-utils", "sha256": "01d9209d6fed20226af0d29b95c5e253a1907b61f1a00153187ac5412f1df9b6", "url": "https://files.pythonhosted.org/packages/63/96/deded55ce54c5e6a2b7dbb790ab9bf7b5be8c7e8cb22e2355108bd8723cd/uuid_utils-0.17.1-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl", "version": "0.17.1"}, {"bytes": 187197, "filename": "websockets-16.1.1-cp313-cp313-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl", "name": "websockets", "sha256": "f5d497865f05bb222cab7016c6034542e84e5f29f49c6fd3f4939cda7197b5b8", "url": "https://files.pythonhosted.org/packages/b5/c2/c676c69444d9db448b3f0a55a98dcc534affce0bce961d9d2f0b8499b10a/websockets-16.1.1-cp313-cp313-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl", "version": "16.1.1"}, {"bytes": 268572, "filename": "xxhash-4.0.1-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl", "name": "xxhash", "sha256": "9a51b061d54cda8b83e62c44458bfbf0dabbef9b975dd9649952ba5076b9f349", "url": "https://files.pythonhosted.org/packages/6f/fb/76580c08e916507859b0f335393cb5fdc59452c4402edbc6bcca6e47e7df/xxhash-4.0.1-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl", "version": "4.0.1"}, {"bytes": 5547173, "filename": "zstandard-0.25.0-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl", "name": "zstandard", "sha256": "8e735494da3db08694d26480f1493ad2cf86e99bdd53e8e9771b2752a5c0246a", "url": "https://files.pythonhosted.org/packages/63/4b/e3678b4e776db00f9f7b2fe58e547e8928ef32727d7a1ff01dea010f3f13/zstandard-0.25.0-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl", "version": "0.25.0"}], "expanded_bytes": 165002177, "helper_sha256": "224c21137a7c9011ce26fd37d6031b894ee2fc06393aa6f3cfcda756be33973f", "kind": "canonical-m5-source", "members": {"AGENTS.md": {"bytes": 5368, "sha256": "adfb95da182d27e526976a1e3d62d27d759a1e025daa85b0e13586c100bcb6d0"}, "PROJECT_SPECIFICATION.md": {"bytes": 63797, "sha256": "37e5e028574eac7815cd4b119ebbc99857e63fc9eb79b25a1b157a392f42a5d6"}, "README.md": {"bytes": 53363, "sha256": "35238797bb1df972597f27c9d1b3caf94630e36883a51b3274cade56fc44e617"}, "SOURCE_COMMIT.txt": {"bytes": 41, "sha256": "1cb45ab2ff376f81b6238f99c8a22ca5d35abb9a095f52200c6d34ba7922954f"}, "configs/study.toml": {"bytes": 175, "sha256": "ce5592c03fbedf77fed8cdce76fb6c78c7732f7366fa8a4614493ec29feee6ba"}, "metadata/4B/chat_template.jinja": {"bytes": 7756, "sha256": "a4aee8afcf2e0711942cf848899be66016f8d14a889ff9ede07bca099c28f715"}, "metadata/4B/config.json": {"bytes": 3161, "sha256": "ddc63e1c717afa86c865bb5e01313d89d72bb53b97ad4a8a03ba8510c0621670"}, "metadata/4B/model.safetensors.index.json": {"bytes": 76196, "sha256": "cf3f798ee02ba45f9622aa8892a47369ab667d0afbf154ee7c2212de42e6302d"}, "metadata/4B/tokenizer.json": {"bytes": 12807982, "sha256": "5f9e4d4901a92b997e463c1f46055088b6cca5ca61a6522d1b9f64c4bb81cb42"}, "metadata/4B/tokenizer_config.json": {"bytes": 16710, "sha256": "316230d6a809701f4db5ea8f8fc862bc3a6f3229c937c174e674ff3ca0a64ac8"}, "metadata/9B/chat_template.jinja": {"bytes": 7756, "sha256": "a4aee8afcf2e0711942cf848899be66016f8d14a889ff9ede07bca099c28f715"}, "metadata/9B/config.json": {"bytes": 3126, "sha256": "d0883072e01861ed0b2d47be3c16c36a8e81c224c7ffaa310c6558fb3f932b05"}, "metadata/9B/model.safetensors.index.json": {"bytes": 79657, "sha256": "26d3539b516be613f39563617cb9d33b3f83d401298125be392c80cefb8f7fe5"}, "metadata/9B/tokenizer.json": {"bytes": 12807982, "sha256": "5f9e4d4901a92b997e463c1f46055088b6cca5ca61a6522d1b9f64c4bb81cb42"}, "metadata/9B/tokenizer_config.json": {"bytes": 16710, "sha256": "316230d6a809701f4db5ea8f8fc862bc3a6f3229c937c174e674ff3ca0a64ac8"}, "pyproject.toml": {"bytes": 644, "sha256": "b552fe08d37bb0f4292593ab1c1ebbb9e90b1e88375de0408ce2eeb924038ae5"}, "recovery/__init__.py": {"bytes": 82, "sha256": "8d142a632ef94a32032fb966f8d1e9253fb4f886ed6b3f745e611f6f3a74fb0f"}, "recovery/__main__.py": {"bytes": 15300, "sha256": "ac890a5ea914636c6decd3a44b5aa7dd43717a5af39da138eb170cd2bb2be2fd"}, "recovery/accounting.py": {"bytes": 29277, "sha256": "081ff337a1e6899f805f0f296a82b3e06ace62993effbc3d749bfe6854bcc3e9"}, "recovery/analysis.py": {"bytes": 31933, "sha256": "32993bbd2e2718b60f428517780fa84b1b924483a564d009d5e12086052be4f1"}, "recovery/attempts.py": {"bytes": 11879, "sha256": "c07c7b02f7f6faea48d3b5d599ca381731729252130771f85d8132df95c34a8f"}, "recovery/costs.py": {"bytes": 35902, "sha256": "e4334dac8fc18bd79134f5cc8b034e19de49f6113bfd780c14b2e64923e26ce3"}, "recovery/edits.py": {"bytes": 4930, "sha256": "4e46f90db9496a24c79186a5e5a6131ea494d27d2c5baa6116c41af293d8cfac"}, "recovery/evaluate.py": {"bytes": 39863, "sha256": "a7d7300579d365ce164f8b274ea42e23d82e1da23ef1cbf1463d8f94eb7fb562"}, "recovery/events.py": {"bytes": 5515, "sha256": "5a8398b12683bd849a2aa6298ac4ac540dd49c279b744beae04974c9b4fa32f0"}, "recovery/figures.py": {"bytes": 14554, "sha256": "6fed46016924c73c5306309dd211d748e6361fb4360a450ac5803a9251dde975"}, "recovery/inference.py": {"bytes": 16180, "sha256": "29d39f3846485f8610b328337035d6e86eba96ccab36f70a28be726ad94aabc4"}, "recovery/information.py": {"bytes": 41962, "sha256": "6cb7c444023a04e07c32136005bfffa8aa7ef0e629616af886f10058abe513e8"}, "recovery/manifests.py": {"bytes": 6954, "sha256": "d5118ccb77bb61218de3c0cdaebe2545a3cf8c42246312e2f1abe29b9db429a2"}, "recovery/populations.py": {"bytes": 14383, "sha256": "263bf51de854136374c1aab7d65a7b214d47a679428411c4db06b4751dc923ae"}, "recovery/prompts.py": {"bytes": 14281, "sha256": "e0e3947f3a1e646d38e6484255aa2809b2c70e27101894f6c02666490ae4f121"}, "recovery/protocol.py": {"bytes": 10913, "sha256": "7243040218e8d8c80215664c7a7237a754642403f97729b637cb4fd725cf46c4"}, "recovery/replay.py": {"bytes": 8067, "sha256": "bdb28a3f54cf3b82ec3cbf25748a18e208a7b3eae92b635fa2caf8a6c3bc6a66"}, "recovery/report.py": {"bytes": 39008, "sha256": "a7792b1f075971b34687accaeb4ab70f35d527954dc80ebe974e5cebbd49418a"}, "recovery/rounds.py": {"bytes": 22618, "sha256": "164d13f20f73b7a976b274fc960e414736e7918a6a2de5302c7c0d2ceb70154a"}, "recovery/runner.py": {"bytes": 15863, "sha256": "4994fc11ce84640699d107f56dc33adecbabf27388aa54dc3d04f2bc669948cf"}, "recovery/schedule.py": {"bytes": 14633, "sha256": "6ae898cbfce87c5d71380695862053c5402a2bca061f88188616d0f0486a4449"}, "recovery/state.py": {"bytes": 22315, "sha256": "1c0722d500c853addd891e1a9507c6c58b8129ac5cd9fbe38927904e2705badd"}, "recovery/storage.py": {"bytes": 33668, "sha256": "b08c68e6d31dc11cb09fa9b98145b0951091314064fbfda25ce4724fd8291cc0"}, "recovery/tasks.py": {"bytes": 10986, "sha256": "370ea734a28aabda922577aab450b5d9f2b1cfcccb7ab64710b81d30874c7901"}, "recovery/truth.py": {"bytes": 4483, "sha256": "af9a5b4f88814ca8f26f9bdd4b6b9fcd5838f1ee4f4c8a9f14bdba7773d845c8"}, "recovery/uncertainty.py": {"bytes": 41134, "sha256": "f674968b29434c2a182aa81c9d260b39d48744b29f5da60f8ce8736d7f2af054"}, "recovery/witness.py": {"bytes": 3467, "sha256": "18a88a78d5f61d2cc8533bdb67e197a875d817afd11810e23e8a1a6ceee88e6d"}, "recovery/workers.py": {"bytes": 33461, "sha256": "47087877401cbf35ec27749071218ae44e1dd39cf4cf85d3b42541abc90972cd"}, "tests/__init__.py": {"bytes": 47, "sha256": "0ba59e0dc0f396b357b97afb9cd489f07d8b85b328b9121985c1074b02abecd0"}, "tests/accounting_archive.py": {"bytes": 4007, "sha256": "cd8517610ca291b0a110e605a74df7a0b3409ec1a760e6ed5229923c169f23ba"}, "tests/accounting_checks.py": {"bytes": 37326, "sha256": "4786a34f20535e05c2ed3e66d698784bad06b2d470ba1772358696c25532437e"}, "tests/attainability_checks.py": {"bytes": 8825, "sha256": "f217f1fd0edc02863dc7fad83626d47b287c0ef267313ac5bb029e6392356285"}, "tests/attempt_archive.py": {"bytes": 3444, "sha256": "09a29b2d7c7a26ab6ee0c5c8c5a9490aaefc154657c602351888012224e50927"}, "tests/attempt_checks.py": {"bytes": 33818, "sha256": "983e5c78fdf76e90b4e0a5095e4a641d4252bb43027d5136c6983d72092e097b"}, "tests/attempt_worker.py": {"bytes": 2440, "sha256": "7d7cbedf9f6d7b09b530bc590d6c437929006ce59ac1f9826bb341a9e139cd57"}, "tests/capacity_bound.py": {"bytes": 8899, "sha256": "c9cbf4de50b2db22b9af92368b6c4933e63c5ad56c85424c276de8d424131c8c"}, "tests/cost_archive.py": {"bytes": 7431, "sha256": "6b7bd50008c0531955a9893745dd442452f2ba95e9f113444ec8dacc143110cd"}, "tests/cost_fixture.py": {"bytes": 3531, "sha256": "046654996ee75dd190d5e85c260dd83e1dc4f7cd7a4eda4c4f10bf7e0509ca3b"}, "tests/durable_archive.py": {"bytes": 8243, "sha256": "045ec745554b79028299ffec7840c65eb5bd48fac824067d6ffaddc38bce066b"}, "tests/durable_checks.py": {"bytes": 29461, "sha256": "d7442d2bcd0f7fcad47a7c7664a794b280906900cbb3471c0268567a94c2a16d"}, "tests/estimand_archive.py": {"bytes": 4584, "sha256": "ea9a9acfbb3bb4ec3c1aa1fa924ad0fb0915d52da7ba23131cdcba0278af7dfa"}, "tests/estimand_fixture.py": {"bytes": 4637, "sha256": "f73292983cdad2db6fdfb0c103179c84288d4d2d6c61618ad20d9415b94aab7b"}, "tests/event_checks.py": {"bytes": 29542, "sha256": "e229394b7e98f24ee238ef9f50cadf914ce26d476fb4265be40e1ea8a8aac3af"}, "tests/fixtures/README.md": {"bytes": 62252, "sha256": "5d67131c1a5ba734ee07b67cf76a808cb890ecc21ced5a45ecc5b9957bb00d86"}, "tests/fixtures/accounting_progress_traces.json.gz": {"bytes": 18722, "sha256": "a290b45664c7afa508aa379669ec581a9d938f7adcac26625b1add2b3431ba48"}, "tests/fixtures/attainability_traces.json.gz": {"bytes": 713869, "sha256": "19a6135c1760ca865ced254be085f879700a6e127be97b595c16e0aa0046cd6c"}, "tests/fixtures/attempt_failure_traces.json.gz": {"bytes": 91551, "sha256": "75cd583d1b1c27e02ef79f297c6a134ef3a37ed1f34e6b41b918de52117ce639"}, "tests/fixtures/capacity_report.json.gz": {"bytes": 316146, "sha256": "e5b0843c9ce1c6a9deef81bf9d2af0b8af041ccbe867ebf31cf970a5f57a9871"}, "tests/fixtures/capacity_report_8192_original.json.gz": {"bytes": 109933, "sha256": "e16d784a8860adcbd42dd69d128b9de11827cac4530b293d18a87aadac77bb7b"}, "tests/fixtures/durable_native_traces.json.gz": {"bytes": 2804533, "sha256": "16ea347f5b1581a61f05ac77040de85ba580d27d37d88b444a9158e01f02af31"}, "tests/fixtures/durable_round_traces.json.gz": {"bytes": 1681343, "sha256": "5398feb8252e9d70739abfcafdd07f7f3b28fc02f8809c74e9085b989157bc78"}, "tests/fixtures/estimand_tables.json.gz": {"bytes": 4290738, "sha256": "b3fcd11b8aecff5e1193e28cea70ee270fc35b3fbefa443140924b49e3edf8bb"}, "tests/fixtures/event_boundaries.json": {"bytes": 2462271, "sha256": "bf9fc8316330137000b7b6c8603cfdc8e53f8fd438bedeb7468cac55d9ea7e22"}, "tests/fixtures/exact_outcome_tables.json.gz": {"bytes": 7378461, "sha256": "5d656cbde86b6debdea39f5a5c2ee6823cec760c1e31e1e8436bf0ed97b7e9c0"}, "tests/fixtures/inference_suffixes.json": {"bytes": 9532, "sha256": "df65d796e4aa2a64e3817cbc501da47798137f368f77298c3e160a787b880371"}, "tests/fixtures/information_tables.json.gz": {"bytes": 980421, "sha256": "48fdb78088da161ccabb5e9fe61d51046dffae9c866c506df5bf46df88621386"}, "tests/fixtures/information_tokens.json": {"bytes": 21035, "sha256": "3a69addeffa9868765e16abbc321f2d6605557c8c2f6c9ae76b16fa5f34e0a14"}, "tests/fixtures/peer_requests.json": {"bytes": 131607, "sha256": "7344eb31287b2066b01da778ad4dd5943c0398f189d74c8d0a2da68f28392b02"}, "tests/fixtures/policy_cost_tables.json.gz": {"bytes": 1283743, "sha256": "44c62787365a0164bf397bdad0b1c1e555ceae3dd62a28acba41f1f02a01a045"}, "tests/fixtures/population_fallback.json": {"bytes": 190832, "sha256": "9b15339a018626109334172edd262d54c3989228981954f068d13ade402a7960"}, "tests/fixtures/population_target.json": {"bytes": 230742, "sha256": "95dfa2b076c37823972cfc94fe9bccb4319d4e09c8936b7a24aa0f526f2fabd1"}, "tests/fixtures/protocol_replies.json": {"bytes": 34110, "sha256": "f4878683e1e4f41dd057dbbe420e6326aa31c36453f04e3b0fd0dc217d964381"}, "tests/fixtures/replay_traces.json": {"bytes": 14929488, "sha256": "85df88dcda5c830a3405839f40bb8c8c584ce120a96c01373564ce2824cd8fd0"}, "tests/fixtures/report_tables.json.gz": {"bytes": 221993, "sha256": "3ecaf83af889d7b350bfaba83e813a671869ade13b5c6606a76606f14934a61c"}, "tests/fixtures/round_traces.json": {"bytes": 1239418, "sha256": "6bed33298b59b5fbd199dfa545e2c8b5527633a6d90a74b50498af31a80d2b87"}, "tests/fixtures/schedules_target.json": {"bytes": 2940008, "sha256": "43d94c144fad6047f7c144927f7728fdb91a5548c85e10922ca94f4acde23e3b"}, "tests/fixtures/scripted_lifecycle_run.zip": {"bytes": 66344034, "sha256": "c3d8606f75107f7bcfb4327bc412cfbeea20f730119e26379deac9331e4cbc22"}, "tests/fixtures/scripted_reports.zip": {"bytes": 19775748, "sha256": "ee6ef6259be4870740a6ed359af97fd78677159ee7effe9dc1b72e32c3bc36bd"}, "tests/fixtures/silo_selected.json": {"bytes": 29835, "sha256": "f34a222a45a04763a6174f704b7e6819ba54990d0cdb3c9483be7e51f409244a"}, "tests/fixtures/uncertainty_tables.json.gz": {"bytes": 7546184, "sha256": "ad6819e68a8c8d22d494590c70c6934db75a0763709df967ef495b578f75624d"}, "tests/fixtures/worker_scheduling_traces.json.gz": {"bytes": 1805767, "sha256": "f86f0efa261fbc221fca6ba303cfe8ef456734744549e9865730cbdeab5ac8d6"}, "tests/inference_checks.py": {"bytes": 18025, "sha256": "209205dc40118986471b841f5f0376a4871ee61deb113968e3592ccc5ca08f2a"}, "tests/inference_mocks.py": {"bytes": 6749, "sha256": "f572a2429af254f7d8f58e14be905a914bba4fcb197bac149e42264562982648"}, "tests/information_archive.py": {"bytes": 5102, "sha256": "0c3b106bbb2128f0c1c9d4bfc12e2f2273de52eeadc5cdf1c3329044321f185c"}, "tests/information_examples.py": {"bytes": 3867, "sha256": "f20a3531faafe031cee7febcda2ec1dadaf296aa6bf175b01570f5a2e18cab89"}, "tests/information_fixture.py": {"bytes": 7364, "sha256": "b0e7ec51b4a4fdb74c9f61eece7b867142e663ed625b5188c27855b73b627bd3"}, "tests/lifecycle_archive.py": {"bytes": 11228, "sha256": "42bf67b16a79bd0a3c88e17fb4865c1437527c68eb4c9708434917ab6682a485"}, "tests/lifecycle_checks.py": {"bytes": 30938, "sha256": "23d357e715dad09bd4118fb593eb08da2cb68015e47e4f58d7186b3a9de3aec2"}, "tests/lifecycle_fixture.py": {"bytes": 7897, "sha256": "6931ec75098bdfd961ac4a375b15d55f14edbb8e96ded4f3e91b6f944e8e3d8f"}, "tests/lifecycle_worker.py": {"bytes": 6116, "sha256": "d59c5b12a7a091db331d94fe52f3b1b96cb7d6fc9e944899cfa9de0e96dd18d8"}, "tests/outcome_archive.py": {"bytes": 5551, "sha256": "7a7f03a100abae1527a08564a27a1b5ffdd50703cd9c72b5d34bf45ae57757a7"}, "tests/outcome_fixture.py": {"bytes": 6409, "sha256": "02c6b4ecd9c48434b9469ad2c8e8440347aaf9386f3bb82e9a2167ad3353d71a"}, "tests/peer_view_cases.py": {"bytes": 3728, "sha256": "dcc5d55134db5586e76b8e5c70ecc9c1139a635f3dc3988148532755ab21442b"}, "tests/protocol_checks.py": {"bytes": 27474, "sha256": "f84f814b725f40efa6923796129eab417cc040ef6fa95078152f2c89d66ecb55"}, "tests/references.py": {"bytes": 1448, "sha256": "6e66c07fca9b84ebf4c999c98855953a8128395c8f1d00184826be29efc01b7e"}, "tests/replay_checks.py": {"bytes": 28401, "sha256": "2b73cfd48437de080ed07279ac41fb84d9cf21445dd55ad668f125903831be06"}, "tests/report_archive.py": {"bytes": 8025, "sha256": "88430ec6cbfb36a11b32d717f785e4ebb162a3e87ca42adee15e80b52d545f54"}, "tests/round_checks.py": {"bytes": 29276, "sha256": "7c43437a74cf784788fa8d7818527baecd5156e046f074c0ef1550e8bcd727f2"}, "tests/test_analysis.py": {"bytes": 24171, "sha256": "52835f58fff9d756318d30c9c2bf44d328eeb4b76ec045007a00e2ec247f9d74"}, "tests/test_analysis_run.py": {"bytes": 7828, "sha256": "9b9da2f0a511a8a7690d8b2c423864f17caf6fc0db4d3a7ca9bc84350abca731"}, "tests/test_cost_archive.py": {"bytes": 2291, "sha256": "77fc81f95c8bb6e47a2850b52efa6470263ce81ffc855f6b55db085427011417"}, "tests/test_costs.py": {"bytes": 19570, "sha256": "f649915c9c882c33c0f2ace5b02de8b3fcf290a0371740a007b3e41975cfd601"}, "tests/test_costs_run.py": {"bytes": 6013, "sha256": "e55877ec08b832b5b282be7d550eaf8d54b2bab3a8b927ed709110c2679b5e8c"}, "tests/test_cpu_entrypoint.py": {"bytes": 12021, "sha256": "23c44d794a79b14cda0377afb7a127ad8e2c4aa58bb4473e3255a5d3e1c0d916"}, "tests/test_edits.py": {"bytes": 8072, "sha256": "923b2e4938b8c4504ff02e31556a28799ee3cf9687c18502f6cca577a208e3bf"}, "tests/test_estimand_archive.py": {"bytes": 1931, "sha256": "e478201976cd9728be466fa7a8a9ba85810e1a094ccf2b6cf3a307bae2478145"}, "tests/test_evaluate.py": {"bytes": 26693, "sha256": "6ca2ec75bf9bd5661eb0fddf161754430d58acf603ce651c232c1c3a2b193853"}, "tests/test_inference_preparation.py": {"bytes": 7566, "sha256": "32f6f6dcf2f017964ac1588e7eefe553ea728a51ca1569352885fe68c418e47f"}, "tests/test_information.py": {"bytes": 14391, "sha256": "7a7d322d87442556f7daac2fd8c53e8a578dbfab94a0180cfa7d86f8103a9a5f"}, "tests/test_information_run.py": {"bytes": 17193, "sha256": "52de60d88bb5f101dd3556c0f6fd55dea3693a259e5270febcb082fb85012c4e"}, "tests/test_outcome_archive.py": {"bytes": 2189, "sha256": "d4cfde0e117bd0f8f605f62a2099b7dfd7ed8557ca4a9a39b2120fa967598737"}, "tests/test_peer_views.py": {"bytes": 29677, "sha256": "e34668dbc3d4d0ed4c826849b98f4c8b3e11b513adcdc851c1d5b0a5927af792"}, "tests/test_populations.py": {"bytes": 18916, "sha256": "37514f75891f3142a6a366d3f70ee771ac4050e8b599dc835e2590d0f123136a"}, "tests/test_report.py": {"bytes": 5571, "sha256": "55bcec70084e648714d9ddd5fe105fb118de7b92d7e600cb1634340b8ff75a99"}, "tests/test_report_archive.py": {"bytes": 5274, "sha256": "06e9af78da397aa5f2fafa7877a4cb240ef66cb3297c1aaef9e3901fd8de0462"}, "tests/test_report_figures.py": {"bytes": 2625, "sha256": "8cb56e74e49e9d8364004444faa8d330bc616997400a4d8d95cc7108f399a785"}, "tests/test_report_run.py": {"bytes": 13622, "sha256": "e48c4ebfa97db2de4935243b253bfd119ee271749f00214878477ffe91e90b1a"}, "tests/test_schedule_manifests.py": {"bytes": 12070, "sha256": "f46362f5b6603810f9dec400f03a2a77c1f95068af4e9ae74137c8a0a625ce65"}, "tests/test_schedules.py": {"bytes": 15733, "sha256": "888eded3e564fb5478894e4fccfb6efd1c8a1069bf789a5794e9aa846e96643a"}, "tests/test_tasks.py": {"bytes": 13501, "sha256": "6ec45a326258af8705fd8da6c2b2fa6dbf8548cfda549b8d128754375367682c"}, "tests/test_truth_contracts.py": {"bytes": 16097, "sha256": "03a9b658dd18e741e098018ba4c22acf63f9c355b511058925e64dc0377818f7"}, "tests/test_uncertainty.py": {"bytes": 28779, "sha256": "bb7598abe9b43c4494b04a1339f93f2222e845cb6a1eaa3ea658552355e496d3"}, "tests/test_uncertainty_archive.py": {"bytes": 3049, "sha256": "9fb06054a1ecc57059e542906a25e71770a4811de84fa383d3b76ebbf00c3c1a"}, "tests/test_uncertainty_run.py": {"bytes": 7017, "sha256": "b13d6d010448951758821608e76760f6e10f51c0ebbcbd257f1162aea9ed81f8"}, "tests/tokenizer_attainability.py": {"bytes": 14840, "sha256": "1839869a7fb2a6cff71cd29a721d0fb2246cf995893c0e19d7b98ef1d80021c6"}, "tests/tokenizer_capacity.py": {"bytes": 27474, "sha256": "62d6b852933c27f1bd0a71bdfbd5f05f6c6e90615a109f84b9715e94f5a14585"}, "tests/tokenizer_events.py": {"bytes": 11113, "sha256": "5e00f08e0dc7471398a58394ede8849865375ac4d9a875ff4a516adbb78f4ac0"}, "tests/tokenizer_inference.py": {"bytes": 4223, "sha256": "447e10925755193441dc1b7a2631bac36384f8c3bde035eb02955037dfa5e538"}, "tests/tokenizer_information.py": {"bytes": 5088, "sha256": "f5d20817f8a96ed03a424da010b404fb06012fc9533df106d63f2e26490ce852"}, "tests/tokenizer_peer_views.py": {"bytes": 8064, "sha256": "7e1a5046d17270210a8e13a8ca12840471ce4a62fda39bb4f2f87ce6866ba68c"}, "tests/tokenizer_protocol.py": {"bytes": 14630, "sha256": "98f98f0d7d2b9207178d575147f27e2bfcd9df3222a21756b0eb9a827a5a4502"}, "tests/tokenizer_replay.py": {"bytes": 14881, "sha256": "8ad88caf6664447c655afec944813ebe3562861e5160e4b04791816c74d35799"}, "tests/tokenizer_rounds.py": {"bytes": 8794, "sha256": "440265cfd25616dd4a17991297e159c720243cd80acddd509c1580e5432ed563"}, "tests/uncertainty_archive.py": {"bytes": 5758, "sha256": "aed89e7cc66abc5586d231ff2b8544620f369911ad66a6d00fa9cb1f88ba5ae4"}, "tests/worker_archive.py": {"bytes": 4288, "sha256": "d46dbb81d713be34dbd416628c53c099bca0bccf2196828c1cc6421e98e63627"}, "tests/worker_checks.py": {"bytes": 26678, "sha256": "5cc43ca3c1ce5596e33ae9f10c452f42b3c767125b0a1447a0e521f957939aef"}, "tests/worker_fixture.py": {"bytes": 2302, "sha256": "5a2ca7d991f26d5cca1a2c6bc33a30aed4eeda64200febdd600f448d9a150a40"}}, "payload_filename": "m5-kaggle-source.payload", "schema_version": 1, "source_baseline_commit": "ca46bcd55486b38bb453f5294e01f943bc7f7752", "transport": "raw_payload"}')
CANDIDATE_LOCK='accelerate==1.14.0 --hash=sha256:e94390c2863b873be18f623f9df48a0d8fe5eff13ea7f1a00092b0a7904888c6\naiosqlite==0.22.1 --hash=sha256:21c002eb13823fad740196c5a2e9d8e62f6243bd9e7e4a1f87fb5e44ecb4fceb\nannotated-doc==0.0.5 --hash=sha256:117bac03a25ede5df5440e855b32d556049ca169ead221505badf432fed4b101\nannotated-types==0.8.0 --hash=sha256:f072f4d804ea359e4eaf198b1af7a8b0943881a87f31bb764f8bf219bb9419e0\nanyio==4.15.1 --hash=sha256:6152fdbbf9a77fdec97731721bebf7c4c44f7c29b424b0065826173efc7ed101\nbitsandbytes==0.49.2 --hash=sha256:54b771f06e1a3c73af5c7f16ccf0fc23a846052813d4b008d10cb6e017dd1c8c\ncertifi==2026.7.22 --hash=sha256:62f22742b58a1a33014a2b6b706588a8d7e2a88ae7bd1a6ebe8c992928483775\ncharset-normalizer==3.5.2 --hash=sha256:7218e8f32b0956cfcd048fd42d9d5779809745ca1d86113ca56f66e7ae1549c4\nclick==8.5.0 --hash=sha256:255bc9599cf7748b4b1a446ccc735421bd08a2ae529a8b88597d3de5664ee360\ncuda-bindings==12.9.9 --hash=sha256:d399447bf5d929d6f59ec4e2c7b26d11ef21e8be5e88995e41c705db1a7c1363\ncuda-pathfinder==1.8.3 --hash=sha256:e29e59829c297a7a5233bd9cc71094fc5bddbd076951482670178f9eade39b1f\ncuda-toolkit==12.8.1 --hash=sha256:adc7906af4ecbf9a352f9dca5734eceb21daec281ccfcf5675e1d2f724fc2cba\ndistro==1.9.0 --hash=sha256:7bffd925d65168f85027d8da9af6bddab658135b840670a223589bc0c8ef02b2\nfilelock==4.0.12 --hash=sha256:5f17ee83ecee8a6f3e389c75822fb70a1c2f0506b99438a6dffa1d56793588c8\nfsspec==2026.9.0 --hash=sha256:8dd6e646e99ea382bd85f97a45e6b526a442d79423a7dc673f1e2756d05fcb5f\nh11==0.16.0 --hash=sha256:63cf8bbe7522de3bf65932fda1d9c2772064ffb3dae62d55932da54b31cb6c86\nhf-xet==1.7.0 --hash=sha256:2814a6e999d13464c4d679b788cc5d784eb5a4edfc638a31f10e9a11ab531ef8\nhttpcore==1.0.9 --hash=sha256:2d400746a40668fc9dec9810239072b40b4484b640a8c38fd654a024c7a1bf55\nhttpcore2==2.13.1 --hash=sha256:e1e05d4f25f7d7d496bfb96748f6f4b67657b03da069b3a68c36069f3db73d0a\nhttpx==0.28.1 --hash=sha256:d909fcccc110f8c7faf814ca82a9a4d816bc5a6dbfea25d6591d6985b8ba59ad\nhttpx2==2.13.1 --hash=sha256:6dff50fabc270ee5fd25d845d0b078ed20564579744d6d962850975996d2f9a4\nhuggingface-hub==1.19.0 --hash=sha256:1dc72e1f6b4d6df6b30eb72e57d00514ef453d660f04af2b87f0e67267f31ee0\nidna==3.20 --hash=sha256:ab7ae7122974553370f0bdb919e1a960b2cd1bc1ef0276416d896db81c14582c\njinja2==3.1.6 --hash=sha256:85ece4451f492d0c13c5dd7c13a64681a86afae63a5f347908daf103ce6d2f67\njsonpatch==1.33 --hash=sha256:0ae28c0cd062bbd8b8ecc26d7d164fbbea9652a1a3693f3b956c1eae5145dade\njsonpointer==3.2.0 --hash=sha256:3e25c9bcadf5ae20c3420922234eaba1ae36ab9bef64cd72454aeaceb0b29aad\nlangchain-core==1.6.6 --hash=sha256:d9a71e312ab7443701238750e04a57aefc037c4f1ec82837c8a1609d9a14a9b1\nlangchain-protocol==0.0.19 --hash=sha256:4cdf879a492a35980fd859ae792d3c65458ccaae504e183c9a10d7eac1f0720f\nlanggraph==1.2.12 --hash=sha256:95403af7b510de8d79164742f71daaf866c69ca804a8cb72bef2cc1fa1ab9813\nlanggraph-checkpoint==4.2.0 --hash=sha256:0547fd228935a0b758865de3a3d6d7a2537c308895d0f9ab092ce9151b5da942\nlanggraph-checkpoint-sqlite==3.1.1 --hash=sha256:8505c54c94a658080525d7e6780fdd4e0c078ff2566b30d399c02cc9f9af1c63\nlanggraph-prebuilt==1.1.0 --hash=sha256:51e311747d755b751d5c6b39b0c1446124d3a7643d2515017e6714b323508fc9\nlanggraph-sdk==0.4.6 --hash=sha256:57122b02bda41d37a97a5e09eb018aa8c9b3dc85c433880073779bd9324b7eea\nlangsmith==0.14.4 --hash=sha256:ae0d98c2b0f9292ef7e12451d480937d79a0e7dcf2412ba847de6f053391b44a\nmarkdown-it-py==4.2.0 --hash=sha256:9f7ebbcd14fe59494226453aed97c1070d83f8d24b6fc3a3bcf9a38092641c4a\nmarkupsafe==3.0.4 --hash=sha256:434139499bb20b502ed3baa1f169e618f924a97e7a777fea1a49446d80106cf6\nmdurl==0.1.2 --hash=sha256:84008a41e51615a49fc9966191ff91509e3c40b939176e643fd50a5c2196b8f8\nmpmath==1.3.0 --hash=sha256:a0b2b9fe80bbcd81a6647ff13108738cfb482d481d826cc0e02f5b35e5c88d2c\nnetworkx==3.7 --hash=sha256:e3fd2c13a7814cee3746340d8d7f8598a67f16a58bf47fb7f8793fab6efca1b0\nnumpy==2.5.3 --hash=sha256:a5fa86b80fd24bcd1aff83ad23be44ea323de3f787be8f8b15d4a65621e25321\nnvidia-cublas-cu12==12.8.4.1 --hash=sha256:8ac4e771d5a348c551b2a426eda6193c19aa630236b418086020df5ba9667142\nnvidia-cuda-cupti-cu12==12.8.90 --hash=sha256:ea0cb07ebda26bb9b29ba82cda34849e73c166c18162d3913575b0c9db9a6182\nnvidia-cuda-nvrtc-cu12==12.8.93 --hash=sha256:a7756528852ef889772a84c6cd89d41dfa74667e24cca16bb31f8f061e3e9994\nnvidia-cuda-runtime-cu12==12.8.90 --hash=sha256:adade8dcbd0edf427b7204d480d6066d33902cab2a4707dcfc48a2d0fd44ab90\nnvidia-cudnn-cu12==9.19.0.56 --hash=sha256:ac6ad90a075bb33a94f2b4cf4622eac13dd4dc65cf6dd9c7572a318516a36625\nnvidia-cufft-cu12==11.3.3.83 --hash=sha256:4d2dd21ec0b88cf61b62e6b43564355e5222e4a3fb394cac0db101f2dd0d4f74\nnvidia-cufile-cu12==1.13.1.3 --hash=sha256:1d069003be650e131b21c932ec3d8969c1715379251f8d23a1860554b1cb24fc\nnvidia-curand-cu12==10.3.9.90 --hash=sha256:b32331d4f4df5d6eefa0554c565b626c7216f87a06a4f56fab27c3b68a830ec9\nnvidia-cusolver-cu12==11.7.3.90 --hash=sha256:4376c11ad263152bd50ea295c05370360776f8c3427b30991df774f9fb26c450\nnvidia-cusparse-cu12==12.5.8.93 --hash=sha256:1ec05d76bbbd8b61b06a80e1eaf8cf4959c3d4ce8e711b65ebd0443bb0ebb13b\nnvidia-cusparselt-cu12==0.7.1 --hash=sha256:f1bb701d6b930d5a7cea44c19ceb973311500847f81b634d802b7b539dc55623\nnvidia-nccl-cu12==2.28.9 --hash=sha256:485776daa8447da5da39681af455aa3b2c2586ddcf4af8772495e7c532c7e5ab\nnvidia-nvjitlink-cu12==12.8.93 --hash=sha256:81ff63371a7ebd6e6451970684f916be2eab07321b73c9d244dc2b4da7f73b88\nnvidia-nvshmem-cu12==3.4.5 --hash=sha256:042f2500f24c021db8a06c5eec2539027d57460e1c1a762055a6554f72c369bd\nnvidia-nvtx-cu12==12.8.90 --hash=sha256:5b17e2001cc0d751a5bc2c6ec6d26ad95913324a4adb86788c944f8ce9ba441f\norjson==3.13.0 --hash=sha256:cbed5f4c4b88d94bcc36115f4c3bb3aa25da1563a5c3328aa3acebce2b083040\normsgpack==1.12.2 --hash=sha256:39c1bd2092880e413902910388be8715f70b9f15f20779d44e673033a6146f2d\npackaging==26.3 --hash=sha256:d7193f7c8e4e93f444fde0262bf90af30e16fa0ad0ad44cb553c87339b23cd1c\npsutil==7.2.2 --hash=sha256:076a2d2f923fd4821644f5ba89f059523da90dc9014e85f8e45a5774ca5bc6f9\npydantic==2.13.5 --hash=sha256:346a034f080da3755d8e9cb5e00e8b07de1d39e4f6e2c87d8ab7cafa0b269a73\npydantic-core==2.46.5 --hash=sha256:6f7b393a8b3da82f5c1fc0751e6d01ac6c55b93c18226a60bdfba4a724efafd1\npygments==2.21.0 --hash=sha256:2363c69b61c4a97c838da3b130dcd6468f4848992b21a82f2a63ec34377137d9\npyyaml==6.0.3 --hash=sha256:0f29edc409a6392443abf94b9cf89ce99889a1dd5376d94316ae5145dfedd5d6\nregex==2026.9.29 --hash=sha256:d49c18f1ea294cf4adde2e5ac256e98c82ea9d708462ce4bf799dffa7cfe8a2c\nrequests==2.34.2 --hash=sha256:2a0d60c172f83ac6ab31e4554906c0f3b3588d37b5cb939b1c061f4907e278e0\nrequests-toolbelt==1.0.0 --hash=sha256:cccfdd665f0a24fcf4726e690f65639d272bb0637b9b92dfd91a5568ccf6bd06\nrich==15.0.0 --hash=sha256:33bd4ef74232fb73fe9279a257718407f169c09b78a87ad3d296f548e27de0bb\nsafetensors==0.8.0 --hash=sha256:fd6f3f93c9a0a7cc2788ee63fb763353d4bd2e89b0751bc78fcf7dda00bea774\nsetuptools==81.0.0 --hash=sha256:fdd925d5c5d9f62e4b74b30d6dd7828ce236fd6ed998a08d81de62ce5a6310d6\nshellingham==1.5.4 --hash=sha256:7ecfff8f2fd72616f7481040475a65b2bf8af90a56c89140852d1120324e8686\nsniffio==1.3.1 --hash=sha256:2f6da418d1f1e0fddd844478f41680e794e6051915791a034ff65e5f100525a2\nsqlite-vec==0.1.9 --hash=sha256:1515727990b49e79bcaf75fdee2ffc7d461f8b66905013231251f1c8938e7786\nsympy==1.14.0 --hash=sha256:e091cc3e99d2141a0ba2847328f5479b05d94a6635cb96148ccb3f34671bd8f5\ntenacity==9.2.1 --hash=sha256:9e56f17539296baab7beabb08b92f6ee3d7be92d8be72d763360677c2ad6580e\ntokenizers==0.22.2 --hash=sha256:369cc9fc8cc10cb24143873a0d95438bb8ee257bb80c71989e3ee290e8d72c67\ntorch==2.11.0+cu128 --hash=sha256:db964b33c55035a72ab3e2162287af8f1cc276039c65d015740cc88c26dcedf7\ntqdm==4.70.1 --hash=sha256:c293e525e6fef9c20e8728fd4612df02a0aa31bb5fe91ecd93e123b1b7bffa73\ntransformers==5.12.0 --hash=sha256:500be9eb644ede81c3103eee7687fc36d05dd75d1c76686c3820b26396fe7c7c\ntriton==3.6.0 --hash=sha256:0075039ff27765480083b1a109999bf27110f3542f1f9fad95f0f9065a36da79\ntruststore==0.10.4 --hash=sha256:adaeaecf1cbb5f4de3b1959b42d41f6fab57b2b1666adb59e89cb0b53361d981\ntyper==0.25.1 --hash=sha256:75caa44ed46a03fb2dab8808753ffacdbfea88495e74c85a28c5eefcf5f39c89\ntyping-extensions==4.16.0 --hash=sha256:481caa481374e813c1b176ada14e97f1f67a4539ce9cfeb3f350d78d6370c2e8\ntyping-inspection==0.4.4 --hash=sha256:65b8397ba37ccbce054456aaccddfc91e6e3083c92824df348d96ca832f3f147\nurllib3==2.8.0 --hash=sha256:0cf3cae568d36aa9576b28dfb35f11328f1cb974ca7647d9475ebb86c75ac6e3\nuuid-utils==0.17.1 --hash=sha256:01d9209d6fed20226af0d29b95c5e253a1907b61f1a00153187ac5412f1df9b6\nwebsockets==16.1.1 --hash=sha256:f5d497865f05bb222cab7016c6034542e84e5f29f49c6fd3f4939cda7197b5b8\nxxhash==4.0.1 --hash=sha256:9a51b061d54cda8b83e62c44458bfbf0dabbef9b975dd9649952ba5076b9f349\nzstandard==0.25.0 --hash=sha256:8e735494da3db08694d26480f1493ad2cf86e99bdd53e8e9771b2752a5c0246a\n'
assert hashlib.sha256(PREFLIGHT_PROGRAM.encode()).hexdigest()==EXPECTED['helper_sha256']
assert hashlib.sha256(CANDIDATE_LOCK.encode()).hexdigest()==EXPECTED['candidate_lock_sha256']
preflight_namespace={'__name__':'recovery_m5_preflight'}
exec(compile(PREFLIGHT_PROGRAM,'m5_kaggle_preflight.py','exec'),preflight_namespace)


## Run source, package and storage checks


In [2]:
# Overrides support controlled local validation; Kaggle defaults need no edits.
input_root=Path(os.environ.get('M5_PREFLIGHT_INPUT_ROOT','/kaggle/input'))
output_root=Path(os.environ.get('M5_PREFLIGHT_OUTPUT_ROOT','/kaggle/working'))
scratch_root=Path(os.environ.get('M5_PREFLIGHT_SCRATCH_ROOT','/tmp/recovery-m5'))
PREFLIGHT_RESULT=preflight_namespace['run_preflight'](input_root,output_root,scratch_root,EXPECTED,CANDIDATE_LOCK)
print(json.dumps(PREFLIGHT_RESULT['summary'],indent=2))


input_inventory: passed


source: passed


cpu_host: failed — ValueError: candidate_host_requires_linux_x86_64_python313


command_cleanup: blocked — prerequisites: cpu_host


environment: blocked — prerequisites: cpu_host, command_cleanup


...

native_9b: blocked — prerequisites: installed_packages


native_4b: blocked — prerequisites: installed_packages


native_capacity: blocked — prerequisites: installed_packages


filesystem_backup: passed


{
  "preparation_checks_passed": false,
  "root_blocker": "ValueError: candidate_host_requires_linux_x86_64_python313",
  "tests_passed": 0,
  "model_generations": 0,
  "model_smoke_ready": false,
  "gpu_native_diagnostics": null,
  "model_assets_verified": null,
  "model_smoke": null,
  "provider_assignment_utc": null,
  "provider_release_utc": null,
  "observed_cpu_wall_seconds": 4.827999999979511,
  "next_action": "Download the ZIP and stop the session. Return the root-blocker evidence for review; do not rerun or repair fixtures."
}


## Download the evidence and stop


In [3]:
from IPython.display import FileLink,display

print(PREFLIGHT_RESULT['summary']['next_action'])
if PREFLIGHT_RESULT['zip_path']:
    print('Result ZIP SHA-256:',PREFLIGHT_RESULT['zip_sha256'])
    display(FileLink(os.path.relpath(PREFLIGHT_RESULT['zip_path'],Path.cwd())))
else:
    print('Export failed; retain completed JSON/logs in:',PREFLIGHT_RESULT['run_directory'])
    print(PREFLIGHT_RESULT['report'].get('export_error'))
print('Stop the Kaggle session after download. Model smoke remains unexecuted.')


Download the ZIP and stop the session. Return the root-blocker evidence for review; do not rerun or repair fixtures.
Result ZIP SHA-256: 342dff4391745b472a7c2224f038ed223fe2d22508c925a6a42c265409f58ea0


C:\Users\EndritHasani\Downloads\recovery_blueprint\recovery_blueprint\notebooks\validation\m5-full-preparation-kernel-final\valid-source\recovery-m5-preflight-20261008T183940546750Z-79c38707\preflight-results.zip

Stop the Kaggle session after download. Model smoke remains unexecuted.
